### 1. 구글 드라이브 마운트

In [ ]:
# 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

### 2. 각 파일 내의 엑셀 파일 로드

In [ ]:
import os
import pandas as pd

# 기본 경로 설정
base_path = '/content/drive/MyDrive/26 국어정보학/기말과제/'

# 키워드별 디렉토리 정의
keyword_dirs = {
    '화제': os.path.join(base_path, 'CollectedData_2018-2025_화제'),
    '논란': os.path.join(base_path, 'CollectedData_2018-2025_논란'),
    '이슈': os.path.join(base_path, 'CollectedData_2018-2025_이슈')
}

# 로드된 모든 엑셀 파일을 저장할 딕셔너리
# 구조: {키워드: {파일이름: DataFrame}}
all_excel_files = {}

print("엑셀 파일 로딩 시작...")

for keyword, dir_path in keyword_dirs.items():
    if os.path.exists(dir_path):
        print(f"\n디렉토리 '{dir_path}'에서 파일 목록을 가져옵니다.")
        # 해당 키워드에 대한 딕셔너리 초기화
        all_excel_files[keyword] = {}

        # 디렉토리 내 파일 목록을 순회
        for filename in os.listdir(dir_path):
            # .xlsx 또는 .xls 확장자를 가진 파일만 처리
            if filename.endswith(('.xlsx', '.xls')):
                file_path = os.path.join(dir_path, filename)
                try:
                    df = pd.read_excel(file_path)
                    all_excel_files[keyword][filename] = df
                    print(f"  - '{filename}' (키워드: {keyword}) 로드 완료. ({df.shape[0]} 행, {df.shape[1]} 열)")
                except Exception as e:
                    print(f"  - 오류: '{filename}' (키워드: {keyword}) 로드 중 문제 발생: {e}")
    else:
        print(f"\n오류: 디렉토리 '{dir_path}'를 찾을 수 없습니다. 경로를 확인해주세요.")

print("\n--- 엑셀 파일 로딩 요약 ---")
if not all_excel_files:
    print("로드된 엑셀 파일이 없습니다. 경로와 파일 존재 여부를 확인해주세요.")
else:
    for keyword, files in all_excel_files.items():
        print(f"키워드: '{keyword}', 로드된 파일 수: {len(files)}개")
        for filename, df in files.items():
            print(f"  - 파일명: '{filename}', 데이터프레임 크기: {df.shape}")

print("\n모든 엑셀 파일 로딩 완료. 'all_excel_files' 변수에 저장되었습니다.")


### 3. 메타데이터 파일 생성 및 저장

In [ ]:
import re

# 메타데이터를 저장할 리스트
metadata_list = []

print("메타데이터 추출 시작...")

for keyword, files_dict in all_excel_files.items():
    for filename, df in files_dict.items():
        # 파일명에서 연도 추출 (예: NewsResult_..._20180101-20181231.xlsx 에서 2018 추출)
        year_match = re.search(r'_(\d{4})\d{4}-\d{8}\.xlsx', filename)
        year = int(year_match.group(1)) if year_match else 'N/A'

        # 언론사별 개수 계산
        media_counts = {}
        if '언론사' in df.columns:
            media_counts = df['언론사'].value_counts().to_dict()

        metadata_list.append({
            '키워드': keyword,
            '연도': year,
            '파일명': filename,
            '총_행_수': df.shape[0],
            '총_열_수': df.shape[1],
            '컬럼_이름': list(df.columns),
            '언론사_별_개수': media_counts # 새로 추가된 부분
        })

# 메타데이터 리스트를 DataFrame으로 변환
metadata_df = pd.DataFrame(metadata_list)

print("\n--- 메타데이터 정리 완료 ---")
print(f"총 {len(metadata_df)}개의 파일에 대한 메타데이터가 정리되었습니다.")
print("\n메타데이터 DataFrame 미리보기:")
display(metadata_df.head())

print("\n키워드별, 연도별 데이터 요약:")
# 키워드별, 연도별로 그룹화하여 행 수 합계 및 파일 수 계산
summary_df = metadata_df.groupby(['키워드', '연도']).agg(
    총_파일_수=('파일명', 'count'),
    총_데이터_행_수=('총_행_수', 'sum')
).reset_index()

display(summary_df.sort_values(by=['키워드', '연도']))

print("\n전체 데이터셋 요약:")
print(f"전체 키워드: {metadata_df['키워드'].nunique()}개")
print(f"전체 연도: {metadata_df['연도'].nunique()}년 ({metadata_df['연도'].min()}~{metadata_df['연도'].max()})")
print(f"총 로드된 엑셀 파일 수: {metadata_df.shape[0]}개")
print(f"전체 데이터 행 수 합계: {metadata_df['총_행_수'].sum()}행")

In [ ]:
# 업데이트된 메타데이터 DataFrame을 엑셀 파일로 저장
output_file_path = os.path.join(base_path, 'metadata_summary.xlsx')

try:
    metadata_df.to_excel(output_file_path, index=False)
    print(f"\n업데이트된 메타데이터가 '{output_file_path}'에 성공적으로 저장되었습니다.")
except Exception as e:
    print(f"\n업데이트된 메타데이터 엑셀 파일 저장 중 오류 발생: {e}")

### 4. 각 기사 키워드 포함 문장 실제 추출

In [ ]:
# --- 웹 스크래핑 헬퍼 함수 정의 ---
def fetch_article_content(url, timeout=3):
    """
    뉴스 기사 URL에서 본문 내용을 추출하는 함수.
    속도 최적화 버전
    """

    headers = {
        'User-Agent': 'Mozilla/5.0'
    }

    try:
        response = requests.get(
            url,
            headers=headers,
            timeout=timeout
        )

        response.raise_for_status()

        soup = BeautifulSoup(
            response.text,
            'lxml'
        )

        content_tags = [
            soup.find('div', class_='article_body'),
            soup.find('div', id='articleBodyContents'),
            soup.find('div', id='article-content'),
            soup.find('div', class_='article-view'),
            soup.find('div', class_='news_cnt'),
            soup.find('article', class_='article_content'),
            soup.find('div', itemprop='articleBody')
        ]

        for tag in content_tags:

            if tag:

                for junk_tag in tag.find_all([
                    'script',
                    'style',
                    'span',
                    'figcaption',
                    'em',
                    'strong',
                    'a',
                    'b'
                ]):
                    junk_tag.extract()

                return tag.get_text(
                    separator='\n',
                    strip=True
                )

        return None

    except Exception:
        return None

### 4-1-1. '화제' 키워드, 2018년 기사에서 문장 추출 및 파일 저장

In [ ]:
current_keyword = '화제'
current_year = 2018

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(current_keyword, current_year, all_excel_files)

if df_keyword_year is None:
    print(f"경고: '{current_keyword}' 키워드에 대해 {current_year}년도 파일을 찾을 수 없습니다.")
else:
    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []
    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = df_keyword_year['URL'].dropna().unique().tolist()

    total_urls_in_excel = len(all_unique_urls_in_excel)

    if total_urls_in_excel == 0:
        print(f"경고: '{current_keyword}' 키워드, {current_year}년 파일에 'URL' 컬럼이 없거나 모든 URL이 비어있습니다. 건너뜁니다.")
    else:
        with tqdm(total=total_urls_in_excel, desc=f"'{current_keyword}' ({current_year}) 기사 처리 중") as pbar:
            for url in all_unique_urls_in_excel:
                import io
                import contextlib
                with contextlib.redirect_stdout(io.StringIO()):
                    article_text = fetch_article_content(url)

                if article_text:
                    sentences = find_sentences_with_keyword(article_text, current_keyword)
                    # '언론사'는 해당 URL이 원본 데이터에 여러 번 나타날 경우 불일치할 수 있으므로, 해당 URL과 일치하는 첫 번째 언론사를 사용합니다.
                    media = 'N/A'
                    matching_rows = df_keyword_year[df_keyword_year['URL'] == url]
                    if not matching_rows.empty and '언론사' in matching_rows.columns:
                        first_media = matching_rows['언론사'].dropna().iloc[0] if not matching_rows['언론사'].dropna().empty else 'N/A'
                        media = first_media

                    if sentences:
                        for sentence in sentences:
                            extracted_sentences_data_current_block.append({
                                '키워드': current_keyword,
                                '연도': current_year,
                                '언론사': media,
                                'URL': url,
                                '추출된_문장': sentence
                            })
                else:
                    failed_urls_current_block.append(url) # Store failed URL
                pbar.update(1)
                time.sleep(0.5) # 서버 과부하 방지를 위한 딜레이를 0.1에서 0.5초로 증가

        current_keyword_year_df = pd.DataFrame(extracted_sentences_data_current_block)

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 완료 ---")
        print(f"총 {len(current_keyword_year_df)}개의 '{current_keyword}' 키워드 문장이 {current_year}년 기사에서 추출되었습니다.")
        print("추출된 문장 DataFrame 미리보기:")
        display(current_keyword_year_df.head())

        # URL 처리 결과 요약
        successful_url_fetches = total_urls_in_excel - len(failed_urls_current_block)
        failed_url_fetches = len(failed_urls_current_block)
        failed_percentage = (failed_url_fetches / total_urls_in_excel * 100) if total_urls_in_excel > 0 else 0

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 URL 처리 결과 요약 ---")
        print(f"전체 엑셀 파일 내 고유 URL 개수: {total_urls_in_excel}개")
        print(f"기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): {successful_url_fetches}개")
        print(f"기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): {failed_url_fetches}개")
        print(f"실패율: {failed_percentage:.2f}%")


        # if failed_urls_current_block:
        #     print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL 목록 ({len(failed_urls_current_block)}개) ---")
        #     for failed_url in failed_urls_current_block:
        #         print(failed_url)
        # else:
        #     print(f"\n'{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL은 없습니다.")

        # 키워드별 추출된 데이터 리스트에 추가 (나중에 키워드별 통합 파일을 위해)
        keyword_specific_extracted_data[current_keyword].append(current_keyword_year_df)
        # 모든 키워드의 통합 저장을 위한 리스트에 추가
        all_extracted_sentences_for_combined_save.append(current_keyword_year_df)

        # 현재 키워드-연도 데이터를 별도 엑셀 파일로 저장
        output_keyword_year_path = os.path.join(base_path, f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx')
        try:
            if not current_keyword_year_df.empty:
                current_keyword_year_df[['언론사', '추출된_문장', '키워드', '연도', 'URL']].to_excel(output_keyword_year_path, index=False)
                print(f"'{current_keyword}' 키워드, {current_year}년 문장이 '{output_keyword_year_path}'에 성공적으로 저장되었습니다.")
            else:
                print(f"'{current_keyword}' 키워드, {current_year}년 추출된 문장이 없어 파일을 저장하지 않습니다.")
        except Exception as e:
            print(f"'{current_keyword}' 키워드, {current_year}년 엑셀 파일 저장 중 오류 발생: {e}")

### 4-1-2. '화제' 키워드, 2019년 기사에서 문장 추출 및 파일 저장

In [ ]:
current_keyword = '화제'
current_year = 2019

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(current_keyword, current_year, all_excel_files)

if df_keyword_year is None:
    print(f"경고: '{current_keyword}' 키워드에 대해 {current_year}년도 파일을 찾을 수 없습니다.")
else:
    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []
    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = df_keyword_year['URL'].dropna().unique().tolist()

    total_urls_in_excel = len(all_unique_urls_in_excel)

    if total_urls_in_excel == 0:
        print(f"경고: '{current_keyword}' 키워드, {current_year}년 파일에 'URL' 컬럼이 없거나 모든 URL이 비어있습니다. 건너뜁니다.")
    else:
        with tqdm(total=total_urls_in_excel, desc=f"'{current_keyword}' ({current_year}) 기사 처리 중") as pbar:
            for url in all_unique_urls_in_excel:
                import io
                import contextlib
                with contextlib.redirect_stdout(io.StringIO()):
                    article_text = fetch_article_content(url)

                if article_text:
                    sentences = find_sentences_with_keyword(article_text, current_keyword)
                    # '언론사'는 해당 URL이 원본 데이터에 여러 번 나타날 경우 불일치할 수 있으므로, 해당 URL과 일치하는 첫 번째 언론사를 사용합니다.
                    media = 'N/A'
                    matching_rows = df_keyword_year[df_keyword_year['URL'] == url]
                    if not matching_rows.empty and '언론사' in matching_rows.columns:
                        first_media = matching_rows['언론사'].dropna().iloc[0] if not matching_rows['언론사'].dropna().empty else 'N/A'
                        media = first_media

                    if sentences:
                        for sentence in sentences:
                            extracted_sentences_data_current_block.append({
                                '키워드': current_keyword,
                                '연도': current_year,
                                '언론사': media,
                                'URL': url,
                                '추출된_문장': sentence
                            })
                else:
                    failed_urls_current_block.append(url) # Store failed URL
                pbar.update(1)
                time.sleep(0.5) # 서버 과부하 방지를 위한 딜레이를 0.1에서 0.5초로 증가

        current_keyword_year_df = pd.DataFrame(extracted_sentences_data_current_block)

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 완료 ---")
        print(f"총 {len(current_keyword_year_df)}개의 '{current_keyword}' 키워드 문장이 {current_year}년 기사에서 추출되었습니다.")
        print("추출된 문장 DataFrame 미리보기:")
        display(current_keyword_year_df.head())

        # URL 처리 결과 요약
        successful_url_fetches = total_urls_in_excel - len(failed_urls_current_block)
        failed_url_fetches = len(failed_urls_current_block)
        failed_percentage = (failed_url_fetches / total_urls_in_excel * 100) if total_urls_in_excel > 0 else 0

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 URL 처리 결과 요약 ---")
        print(f"전체 엑셀 파일 내 고유 URL 개수: {total_urls_in_excel}개")
        print(f"기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): {successful_url_fetches}개")
        print(f"기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): {failed_url_fetches}개")
        print(f"실패율: {failed_percentage:.2f}%")


        # if failed_urls_current_block:
        #     print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL 목록 ({len(failed_urls_current_block)}개) ---")
        #     for failed_url in failed_urls_current_block:
        #         print(failed_url)
        # else:
        #     print(f"\n'{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL은 없습니다.")

        # 키워드별 추출된 데이터 리스트에 추가 (나중에 키워드별 통합 파일을 위해)
        keyword_specific_extracted_data[current_keyword].append(current_keyword_year_df)
        # 모든 키워드의 통합 저장을 위한 리스트에 추가
        all_extracted_sentences_for_combined_save.append(current_keyword_year_df)

        # 현재 키워드-연도 데이터를 별도 엑셀 파일로 저장
        output_keyword_year_path = os.path.join(base_path, f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx')
        try:
            if not current_keyword_year_df.empty:
                current_keyword_year_df[['언론사', '추출된_문장', '키워드', '연도', 'URL']].to_excel(output_keyword_year_path, index=False)
                print(f"'{current_keyword}' 키워드, {current_year}년 문장이 '{output_keyword_year_path}'에 성공적으로 저장되었습니다.")
            else:
                print(f"'{current_keyword}' 키워드, {current_year}년 추출된 문장이 없어 파일을 저장하지 않습니다.")
        except Exception as e:
            print(f"'{current_keyword}' 키워드, {current_year}년 엑셀 파일 저장 중 오류 발생: {e}")

### 4-1-3. '화제' 키워드, 2020년 기사에서 문장 추출 및 파일 저장

In [ ]:
current_keyword = '화제'
current_year = 2020

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(current_keyword, current_year, all_excel_files)

if df_keyword_year is None:
    print(f"경고: '{current_keyword}' 키워드에 대해 {current_year}년도 파일을 찾을 수 없습니다.")
else:
    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []
    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = df_keyword_year['URL'].dropna().unique().tolist()

    total_urls_in_excel = len(all_unique_urls_in_excel)

    if total_urls_in_excel == 0:
        print(f"경고: '{current_keyword}' 키워드, {current_year}년 파일에 'URL' 컬럼이 없거나 모든 URL이 비어있습니다. 건너뜁니다.")
    else:
        with tqdm(total=total_urls_in_excel, desc=f"'{current_keyword}' ({current_year}) 기사 처리 중") as pbar:
            for url in all_unique_urls_in_excel:
                import io
                import contextlib
                with contextlib.redirect_stdout(io.StringIO()):
                    article_text = fetch_article_content(url)

                if article_text:
                    sentences = find_sentences_with_keyword(article_text, current_keyword)
                    # '언론사'는 해당 URL이 원본 데이터에 여러 번 나타날 경우 불일치할 수 있으므로, 해당 URL과 일치하는 첫 번째 언론사를 사용합니다.
                    media = 'N/A'
                    matching_rows = df_keyword_year[df_keyword_year['URL'] == url]
                    if not matching_rows.empty and '언론사' in matching_rows.columns:
                        first_media = matching_rows['언론사'].dropna().iloc[0] if not matching_rows['언론사'].dropna().empty else 'N/A'
                        media = first_media

                    if sentences:
                        for sentence in sentences:
                            extracted_sentences_data_current_block.append({
                                '키워드': current_keyword,
                                '연도': current_year,
                                '언론사': media,
                                'URL': url,
                                '추출된_문장': sentence
                            })
                else:
                    failed_urls_current_block.append(url) # Store failed URL
                pbar.update(1)
                time.sleep(0.5) # 서버 과부하 방지를 위한 딜레이를 0.1에서 0.5초로 증가

        current_keyword_year_df = pd.DataFrame(extracted_sentences_data_current_block)

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 완료 ---")
        print(f"총 {len(current_keyword_year_df)}개의 '{current_keyword}' 키워드 문장이 {current_year}년 기사에서 추출되었습니다.")
        print("추출된 문장 DataFrame 미리보기:")
        display(current_keyword_year_df.head())

        # URL 처리 결과 요약
        successful_url_fetches = total_urls_in_excel - len(failed_urls_current_block)
        failed_url_fetches = len(failed_urls_current_block)
        failed_percentage = (failed_url_fetches / total_urls_in_excel * 100) if total_urls_in_excel > 0 else 0

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 URL 처리 결과 요약 ---")
        print(f"전체 엑셀 파일 내 고유 URL 개수: {total_urls_in_excel}개")
        print(f"기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): {successful_url_fetches}개")
        print(f"기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): {failed_url_fetches}개")
        print(f"실패율: {failed_percentage:.2f}%")


        # if failed_urls_current_block:
        #     print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL 목록 ({len(failed_urls_current_block)}개) ---")
        #     for failed_url in failed_urls_current_block:
        #         print(failed_url)
        # else:
        #     print(f"\n'{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL은 없습니다.")

        # 키워드별 추출된 데이터 리스트에 추가 (나중에 키워드별 통합 파일을 위해)
        keyword_specific_extracted_data[current_keyword].append(current_keyword_year_df)
        # 모든 키워드의 통합 저장을 위한 리스트에 추가
        all_extracted_sentences_for_combined_save.append(current_keyword_year_df)

        # 현재 키워드-연도 데이터를 별도 엑셀 파일로 저장
        output_keyword_year_path = os.path.join(base_path, f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx')
        try:
            if not current_keyword_year_df.empty:
                current_keyword_year_df[['언론사', '추출된_문장', '키워드', '연도', 'URL']].to_excel(output_keyword_year_path, index=False)
                print(f"'{current_keyword}' 키워드, {current_year}년 문장이 '{output_keyword_year_path}'에 성공적으로 저장되었습니다.")
            else:
                print(f"'{current_keyword}' 키워드, {current_year}년 추출된 문장이 없어 파일을 저장하지 않습니다.")
        except Exception as e:
            print(f"'{current_keyword}' 키워드, {current_year}년 엑셀 파일 저장 중 오류 발생: {e}")

### 4-1-4. '화제' 키워드, 2021년 기사에서 문장 추출 및 파일 저장

In [ ]:
current_keyword = '화제'
current_year = 2021

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(current_keyword, current_year, all_excel_files)

if df_keyword_year is None:
    print(f"경고: '{current_keyword}' 키워드에 대해 {current_year}년도 파일을 찾을 수 없습니다.")
else:
    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []
    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = df_keyword_year['URL'].dropna().unique().tolist()

    total_urls_in_excel = len(all_unique_urls_in_excel)

    if total_urls_in_excel == 0:
        print(f"경고: '{current_keyword}' 키워드, {current_year}년 파일에 'URL' 컬럼이 없거나 모든 URL이 비어있습니다. 건너뜁니다.")
    else:
        with tqdm(total=total_urls_in_excel, desc=f"'{current_keyword}' ({current_year}) 기사 처리 중") as pbar:
            for url in all_unique_urls_in_excel:
                import io
                import contextlib
                with contextlib.redirect_stdout(io.StringIO()):
                    article_text = fetch_article_content(url)

                if article_text:
                    sentences = find_sentences_with_keyword(article_text, current_keyword)
                    # '언론사'는 해당 URL이 원본 데이터에 여러 번 나타날 경우 불일치할 수 있으므로, 해당 URL과 일치하는 첫 번째 언론사를 사용합니다.
                    media = 'N/A'
                    matching_rows = df_keyword_year[df_keyword_year['URL'] == url]
                    if not matching_rows.empty and '언론사' in matching_rows.columns:
                        first_media = matching_rows['언론사'].dropna().iloc[0] if not matching_rows['언론사'].dropna().empty else 'N/A'
                        media = first_media

                    if sentences:
                        for sentence in sentences:
                            extracted_sentences_data_current_block.append({
                                '키워드': current_keyword,
                                '연도': current_year,
                                '언론사': media,
                                'URL': url,
                                '추출된_문장': sentence
                            })
                else:
                    failed_urls_current_block.append(url) # Store failed URL
                pbar.update(1)
                time.sleep(0.5) # 서버 과부하 방지를 위한 딜레이를 0.1에서 0.5초로 증가

        current_keyword_year_df = pd.DataFrame(extracted_sentences_data_current_block)

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 완료 ---")
        print(f"총 {len(current_keyword_year_df)}개의 '{current_keyword}' 키워드 문장이 {current_year}년 기사에서 추출되었습니다.")
        print("추출된 문장 DataFrame 미리보기:")
        display(current_keyword_year_df.head())

        # URL 처리 결과 요약
        successful_url_fetches = total_urls_in_excel - len(failed_urls_current_block)
        failed_url_fetches = len(failed_urls_current_block)
        failed_percentage = (failed_url_fetches / total_urls_in_excel * 100) if total_urls_in_excel > 0 else 0

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 URL 처리 결과 요약 ---")
        print(f"전체 엑셀 파일 내 고유 URL 개수: {total_urls_in_excel}개")
        print(f"기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): {successful_url_fetches}개")
        print(f"기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): {failed_url_fetches}개")
        print(f"실패율: {failed_percentage:.2f}%")


        # if failed_urls_current_block:
        #     print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL 목록 ({len(failed_urls_current_block)}개) ---")
        #     for failed_url in failed_urls_current_block:
        #         print(failed_url)
        # else:
        #     print(f"\n'{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL은 없습니다.")

        # 키워드별 추출된 데이터 리스트에 추가 (나중에 키워드별 통합 파일을 위해)
        keyword_specific_extracted_data[current_keyword].append(current_keyword_year_df)
        # 모든 키워드의 통합 저장을 위한 리스트에 추가
        all_extracted_sentences_for_combined_save.append(current_keyword_year_df)

        # 현재 키워드-연도 데이터를 별도 엑셀 파일로 저장
        output_keyword_year_path = os.path.join(base_path, f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx')
        try:
            if not current_keyword_year_df.empty:
                current_keyword_year_df[['언론사', '추출된_문장', '키워드', '연도', 'URL']].to_excel(output_keyword_year_path, index=False)
                print(f"'{current_keyword}' 키워드, {current_year}년 문장이 '{output_keyword_year_path}'에 성공적으로 저장되었습니다.")
            else:
                print(f"'{current_keyword}' 키워드, {current_year}년 추출된 문장이 없어 파일을 저장하지 않습니다.")
        except Exception as e:
            print(f"'{current_keyword}' 키워드, {current_year}년 엑셀 파일 저장 중 오류 발생: {e}")

### 4-1-5. '화제' 키워드, 2022년 기사에서 문장 추출 및 파일 저장

In [ ]:
current_keyword = '화제'
current_year = 2022

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(current_keyword, current_year, all_excel_files)

if df_keyword_year is None:
    print(f"경고: '{current_keyword}' 키워드에 대해 {current_year}년도 파일을 찾을 수 없습니다.")
else:
    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []
    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = df_keyword_year['URL'].dropna().unique().tolist()

    total_urls_in_excel = len(all_unique_urls_in_excel)

    if total_urls_in_excel == 0:
        print(f"경고: '{current_keyword}' 키워드, {current_year}년 파일에 'URL' 컬럼이 없거나 모든 URL이 비어있습니다. 건너뜁니다.")
    else:
        with tqdm(total=total_urls_in_excel, desc=f"'{current_keyword}' ({current_year}) 기사 처리 중") as pbar:
            for url in all_unique_urls_in_excel:
                import io
                import contextlib
                with contextlib.redirect_stdout(io.StringIO()):
                    article_text = fetch_article_content(url)

                if article_text:
                    sentences = find_sentences_with_keyword(article_text, current_keyword)
                    # '언론사'는 해당 URL이 원본 데이터에 여러 번 나타날 경우 불일치할 수 있으므로, 해당 URL과 일치하는 첫 번째 언론사를 사용합니다.
                    media = 'N/A'
                    matching_rows = df_keyword_year[df_keyword_year['URL'] == url]
                    if not matching_rows.empty and '언론사' in matching_rows.columns:
                        first_media = matching_rows['언론사'].dropna().iloc[0] if not matching_rows['언론사'].dropna().empty else 'N/A'
                        media = first_media

                    if sentences:
                        for sentence in sentences:
                            extracted_sentences_data_current_block.append({
                                '키워드': current_keyword,
                                '연도': current_year,
                                '언론사': media,
                                'URL': url,
                                '추출된_문장': sentence
                            })
                else:
                    failed_urls_current_block.append(url) # Store failed URL
                pbar.update(1)
                time.sleep(0.5) # 서버 과부하 방지를 위한 딜레이를 0.1에서 0.5초로 증가

        current_keyword_year_df = pd.DataFrame(extracted_sentences_data_current_block)

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 완료 ---")
        print(f"총 {len(current_keyword_year_df)}개의 '{current_keyword}' 키워드 문장이 {current_year}년 기사에서 추출되었습니다.")
        print("추출된 문장 DataFrame 미리보기:")
        display(current_keyword_year_df.head())

        # URL 처리 결과 요약
        successful_url_fetches = total_urls_in_excel - len(failed_urls_current_block)
        failed_url_fetches = len(failed_urls_current_block)
        failed_percentage = (failed_url_fetches / total_urls_in_excel * 100) if total_urls_in_excel > 0 else 0

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 URL 처리 결과 요약 ---")
        print(f"전체 엑셀 파일 내 고유 URL 개수: {total_urls_in_excel}개")
        print(f"기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): {successful_url_fetches}개")
        print(f"기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): {failed_url_fetches}개")
        print(f"실패율: {failed_percentage:.2f}%")


        # if failed_urls_current_block:
        #     print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL 목록 ({len(failed_urls_current_block)}개) ---")
        #     for failed_url in failed_urls_current_block:
        #         print(failed_url)
        # else:
        #     print(f"\n'{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL은 없습니다.")

        # 키워드별 추출된 데이터 리스트에 추가 (나중에 키워드별 통합 파일을 위해)
        keyword_specific_extracted_data[current_keyword].append(current_keyword_year_df)
        # 모든 키워드의 통합 저장을 위한 리스트에 추가
        all_extracted_sentences_for_combined_save.append(current_keyword_year_df)

        # 현재 키워드-연도 데이터를 별도 엑셀 파일로 저장
        output_keyword_year_path = os.path.join(base_path, f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx')
        try:
            if not current_keyword_year_df.empty:
                current_keyword_year_df[['언론사', '추출된_문장', '키워드', '연도', 'URL']].to_excel(output_keyword_year_path, index=False)
                print(f"'{current_keyword}' 키워드, {current_year}년 문장이 '{output_keyword_year_path}'에 성공적으로 저장되었습니다.")
            else:
                print(f"'{current_keyword}' 키워드, {current_year}년 추출된 문장이 없어 파일을 저장하지 않습니다.")
        except Exception as e:
            print(f"'{current_keyword}' 키워드, {current_year}년 엑셀 파일 저장 중 오류 발생: {e}")

### 4-1-6. '화제' 키워드, 2023년 기사에서 문장 추출 및 파일 저장

In [ ]:
current_keyword = '화제'
current_year = 2023

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(current_keyword, current_year, all_excel_files)

if df_keyword_year is None:
    print(f"경고: '{current_keyword}' 키워드에 대해 {current_year}년도 파일을 찾을 수 없습니다.")
else:
    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []
    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = df_keyword_year['URL'].dropna().unique().tolist()

    total_urls_in_excel = len(all_unique_urls_in_excel)

    if total_urls_in_excel == 0:
        print(f"경고: '{current_keyword}' 키워드, {current_year}년 파일에 'URL' 컬럼이 없거나 모든 URL이 비어있습니다. 건너뜁니다.")
    else:
        with tqdm(total=total_urls_in_excel, desc=f"'{current_keyword}' ({current_year}) 기사 처리 중") as pbar:
            for url in all_unique_urls_in_excel:
                import io
                import contextlib
                with contextlib.redirect_stdout(io.StringIO()):
                    article_text = fetch_article_content(url)

                if article_text:
                    sentences = find_sentences_with_keyword(article_text, current_keyword)
                    # '언론사'는 해당 URL이 원본 데이터에 여러 번 나타날 경우 불일치할 수 있으므로, 해당 URL과 일치하는 첫 번째 언론사를 사용합니다.
                    media = 'N/A'
                    matching_rows = df_keyword_year[df_keyword_year['URL'] == url]
                    if not matching_rows.empty and '언론사' in matching_rows.columns:
                        first_media = matching_rows['언론사'].dropna().iloc[0] if not matching_rows['언론사'].dropna().empty else 'N/A'
                        media = first_media

                    if sentences:
                        for sentence in sentences:
                            extracted_sentences_data_current_block.append({
                                '키워드': current_keyword,
                                '연도': current_year,
                                '언론사': media,
                                'URL': url,
                                '추출된_문장': sentence
                            })
                else:
                    failed_urls_current_block.append(url) # Store failed URL
                pbar.update(1)
                time.sleep(0.5) # 서버 과부하 방지를 위한 딜레이를 0.1에서 0.5초로 증가

        current_keyword_year_df = pd.DataFrame(extracted_sentences_data_current_block)

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 완료 ---")
        print(f"총 {len(current_keyword_year_df)}개의 '{current_keyword}' 키워드 문장이 {current_year}년 기사에서 추출되었습니다.")
        print("추출된 문장 DataFrame 미리보기:")
        display(current_keyword_year_df.head())

        # URL 처리 결과 요약
        successful_url_fetches = total_urls_in_excel - len(failed_urls_current_block)
        failed_url_fetches = len(failed_urls_current_block)
        failed_percentage = (failed_url_fetches / total_urls_in_excel * 100) if total_urls_in_excel > 0 else 0

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 URL 처리 결과 요약 ---")
        print(f"전체 엑셀 파일 내 고유 URL 개수: {total_urls_in_excel}개")
        print(f"기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): {successful_url_fetches}개")
        print(f"기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): {failed_url_fetches}개")
        print(f"실패율: {failed_percentage:.2f}%")


        # if failed_urls_current_block:
        #     print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL 목록 ({len(failed_urls_current_block)}개) ---")
        #     for failed_url in failed_urls_current_block:
        #         print(failed_url)
        # else:
        #     print(f"\n'{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL은 없습니다.")

        # 키워드별 추출된 데이터 리스트에 추가 (나중에 키워드별 통합 파일을 위해)
        keyword_specific_extracted_data[current_keyword].append(current_keyword_year_df)
        # 모든 키워드의 통합 저장을 위한 리스트에 추가
        all_extracted_sentences_for_combined_save.append(current_keyword_year_df)

        # 현재 키워드-연도 데이터를 별도 엑셀 파일로 저장
        output_keyword_year_path = os.path.join(base_path, f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx')
        try:
            if not current_keyword_year_df.empty:
                current_keyword_year_df[['언론사', '추출된_문장', '키워드', '연도', 'URL']].to_excel(output_keyword_year_path, index=False)
                print(f"'{current_keyword}' 키워드, {current_year}년 문장이 '{output_keyword_year_path}'에 성공적으로 저장되었습니다.")
            else:
                print(f"'{current_keyword}' 키워드, {current_year}년 추출된 문장이 없어 파일을 저장하지 않습니다.")
        except Exception as e:
            print(f"'{current_keyword}' 키워드, {current_year}년 엑셀 파일 저장 중 오류 발생: {e}")

### 4-1-7. '화제' 키워드, 2024년 기사에서 문장 추출 및 파일 저장

In [ ]:
current_keyword = '화제'
current_year = 2024

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(current_keyword, current_year, all_excel_files)

if df_keyword_year is None:
    print(f"경고: '{current_keyword}' 키워드에 대해 {current_year}년도 파일을 찾을 수 없습니다.")
else:
    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []
    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = df_keyword_year['URL'].dropna().unique().tolist()

    total_urls_in_excel = len(all_unique_urls_in_excel)

    if total_urls_in_excel == 0:
        print(f"경고: '{current_keyword}' 키워드, {current_year}년 파일에 'URL' 컬럼이 없거나 모든 URL이 비어있습니다. 건너뜁니다.")
    else:
        with tqdm(total=total_urls_in_excel, desc=f"'{current_keyword}' ({current_year}) 기사 처리 중") as pbar:
            for url in all_unique_urls_in_excel:
                import io
                import contextlib
                with contextlib.redirect_stdout(io.StringIO()):
                    article_text = fetch_article_content(url)

                if article_text:
                    sentences = find_sentences_with_keyword(article_text, current_keyword)
                    # '언론사'는 해당 URL이 원본 데이터에 여러 번 나타날 경우 불일치할 수 있으므로, 해당 URL과 일치하는 첫 번째 언론사를 사용합니다.
                    media = 'N/A'
                    matching_rows = df_keyword_year[df_keyword_year['URL'] == url]
                    if not matching_rows.empty and '언론사' in matching_rows.columns:
                        first_media = matching_rows['언론사'].dropna().iloc[0] if not matching_rows['언론사'].dropna().empty else 'N/A'
                        media = first_media

                    if sentences:
                        for sentence in sentences:
                            extracted_sentences_data_current_block.append({
                                '키워드': current_keyword,
                                '연도': current_year,
                                '언론사': media,
                                'URL': url,
                                '추출된_문장': sentence
                            })
                else:
                    failed_urls_current_block.append(url) # Store failed URL
                pbar.update(1)
                time.sleep(0.5) # 서버 과부하 방지를 위한 딜레이를 0.1에서 0.5초로 증가

        current_keyword_year_df = pd.DataFrame(extracted_sentences_data_current_block)

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 완료 ---")
        print(f"총 {len(current_keyword_year_df)}개의 '{current_keyword}' 키워드 문장이 {current_year}년 기사에서 추출되었습니다.")
        print("추출된 문장 DataFrame 미리보기:")
        display(current_keyword_year_df.head())

        # URL 처리 결과 요약
        successful_url_fetches = total_urls_in_excel - len(failed_urls_current_block)
        failed_url_fetches = len(failed_urls_current_block)
        failed_percentage = (failed_url_fetches / total_urls_in_excel * 100) if total_urls_in_excel > 0 else 0

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 URL 처리 결과 요약 ---")
        print(f"전체 엑셀 파일 내 고유 URL 개수: {total_urls_in_excel}개")
        print(f"기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): {successful_url_fetches}개")
        print(f"기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): {failed_url_fetches}개")
        print(f"실패율: {failed_percentage:.2f}%")


        # if failed_urls_current_block:
        #     print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL 목록 ({len(failed_urls_current_block)}개) ---")
        #     for failed_url in failed_urls_current_block:
        #         print(failed_url)
        # else:
        #     print(f"\n'{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL은 없습니다.")

        # 키워드별 추출된 데이터 리스트에 추가 (나중에 키워드별 통합 파일을 위해)
        keyword_specific_extracted_data[current_keyword].append(current_keyword_year_df)
        # 모든 키워드의 통합 저장을 위한 리스트에 추가
        all_extracted_sentences_for_combined_save.append(current_keyword_year_df)

        # 현재 키워드-연도 데이터를 별도 엑셀 파일로 저장
        output_keyword_year_path = os.path.join(base_path, f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx')
        try:
            if not current_keyword_year_df.empty:
                current_keyword_year_df[['언론사', '추출된_문장', '키워드', '연도', 'URL']].to_excel(output_keyword_year_path, index=False)
                print(f"'{current_keyword}' 키워드, {current_year}년 문장이 '{output_keyword_year_path}'에 성공적으로 저장되었습니다.")
            else:
                print(f"'{current_keyword}' 키워드, {current_year}년 추출된 문장이 없어 파일을 저장하지 않습니다.")
        except Exception as e:
            print(f"'{current_keyword}' 키워드, {current_year}년 엑셀 파일 저장 중 오류 발생: {e}")

### 4-1-8. '화제' 키워드, 2025년 기사에서 문장 추출 및 파일 저장

In [ ]:
current_keyword = '화제'
current_year = 2025

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(current_keyword, current_year, all_excel_files)

if df_keyword_year is None:
    print(f"경고: '{current_keyword}' 키워드에 대해 {current_year}년도 파일을 찾을 수 없습니다.")
else:
    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []
    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = df_keyword_year['URL'].dropna().unique().tolist()

    total_urls_in_excel = len(all_unique_urls_in_excel)

    if total_urls_in_excel == 0:
        print(f"경고: '{current_keyword}' 키워드, {current_year}년 파일에 'URL' 컬럼이 없거나 모든 URL이 비어있습니다. 건너뜁니다.")
    else:
        with tqdm(total=total_urls_in_excel, desc=f"'{current_keyword}' ({current_year}) 기사 처리 중") as pbar:
            for url in all_unique_urls_in_excel:
                import io
                import contextlib
                with contextlib.redirect_stdout(io.StringIO()):
                    article_text = fetch_article_content(url)

                if article_text:
                    sentences = find_sentences_with_keyword(article_text, current_keyword)
                    # '언론사'는 해당 URL이 원본 데이터에 여러 번 나타날 경우 불일치할 수 있으므로, 해당 URL과 일치하는 첫 번째 언론사를 사용합니다.
                    media = 'N/A'
                    matching_rows = df_keyword_year[df_keyword_year['URL'] == url]
                    if not matching_rows.empty and '언론사' in matching_rows.columns:
                        first_media = matching_rows['언론사'].dropna().iloc[0] if not matching_rows['언론사'].dropna().empty else 'N/A'
                        media = first_media

                    if sentences:
                        for sentence in sentences:
                            extracted_sentences_data_current_block.append({
                                '키워드': current_keyword,
                                '연도': current_year,
                                '언론사': media,
                                'URL': url,
                                '추출된_문장': sentence
                            })
                else:
                    failed_urls_current_block.append(url) # Store failed URL
                pbar.update(1)
                time.sleep(0.5) # 서버 과부하 방지를 위한 딜레이를 0.1에서 0.5초로 증가

        current_keyword_year_df = pd.DataFrame(extracted_sentences_data_current_block)

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 완료 ---")
        print(f"총 {len(current_keyword_year_df)}개의 '{current_keyword}' 키워드 문장이 {current_year}년 기사에서 추출되었습니다.")
        print("추출된 문장 DataFrame 미리보기:")
        display(current_keyword_year_df.head())

        # URL 처리 결과 요약
        successful_url_fetches = total_urls_in_excel - len(failed_urls_current_block)
        failed_url_fetches = len(failed_urls_current_block)
        failed_percentage = (failed_url_fetches / total_urls_in_excel * 100) if total_urls_in_excel > 0 else 0

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 URL 처리 결과 요약 ---")
        print(f"전체 엑셀 파일 내 고유 URL 개수: {total_urls_in_excel}개")
        print(f"기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): {successful_url_fetches}개")
        print(f"기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): {failed_url_fetches}개")
        print(f"실패율: {failed_percentage:.2f}%")


        # if failed_urls_current_block:
        #     print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL 목록 ({len(failed_urls_current_block)}개) ---")
        #     for failed_url in failed_urls_current_block:
        #         print(failed_url)
        # else:
        #     print(f"\n'{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL은 없습니다.")

        # 키워드별 추출된 데이터 리스트에 추가 (나중에 키워드별 통합 파일을 위해)
        keyword_specific_extracted_data[current_keyword].append(current_keyword_year_df)
        # 모든 키워드의 통합 저장을 위한 리스트에 추가
        all_extracted_sentences_for_combined_save.append(current_keyword_year_df)

        # 현재 키워드-연도 데이터를 별도 엑셀 파일로 저장
        output_keyword_year_path = os.path.join(base_path, f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx')
        try:
            if not current_keyword_year_df.empty:
                current_keyword_year_df[['언론사', '추출된_문장', '키워드', '연도', 'URL']].to_excel(output_keyword_year_path, index=False)
                print(f"'{current_keyword}' 키워드, {current_year}년 문장이 '{output_keyword_year_path}'에 성공적으로 저장되었습니다.")
            else:
                print(f"'{current_keyword}' 키워드, {current_year}년 추출된 문장이 없어 파일을 저장하지 않습니다.")
        except Exception as e:
            print(f"'{current_keyword}' 키워드, {current_year}년 엑셀 파일 저장 중 오류 발생: {e}")

### 4-2-1. '논란' 키워드, 2018년 기사에서 문장 추출 및 파일 저장

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

current_keyword = '논란'
current_year = 2018

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(
    current_keyword,
    current_year,
    all_excel_files
)

if df_keyword_year is None:
    print(
        f"경고: '{current_keyword}' 키워드에 대해 "
        f"{current_year}년도 파일을 찾을 수 없습니다."
    )

else:

    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []

    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = (
            df_keyword_year['URL']
            .dropna()
            .unique()
            .tolist()
        )

    initial_total_urls = len(all_unique_urls_in_excel)

    all_unique_urls_to_process = [
        url
        for url in all_unique_urls_in_excel
        if url not in globally_failed_urls
    ]

    skipped_urls_count = (
        initial_total_urls
        - len(all_unique_urls_to_process)
    )

    if skipped_urls_count > 0:
        print(
            f"이전에 실패하여 건너뛴 URL: "
            f"{skipped_urls_count}개"
        )

    total_urls_to_process = len(
        all_unique_urls_to_process
    )

    if total_urls_to_process == 0:

        if initial_total_urls > 0:
            print(
                f"경고: '{current_keyword}' 키워드, "
                f"{current_year}년 파일의 모든 URL이 "
                f"이전에 실패하여 건너뜁니다."
            )
        else:
            print(
                f"경고: '{current_keyword}' 키워드, "
                f"{current_year}년 파일에 URL이 없습니다."
            )

    else:

        # URL → 언론사 매핑
        url_to_media = (
            df_keyword_year
            .dropna(subset=['URL'])
            .drop_duplicates(subset=['URL'])
            .set_index('URL')['언론사']
            .to_dict()
        )

        def process_url(url):

            article_text = fetch_article_content(url)

            if not article_text:
                return {
                    "success": False,
                    "url": url
                }

            sentences = find_sentences_with_keyword(
                article_text,
                current_keyword
            )

            media = url_to_media.get(
                url,
                'N/A'
            )

            results = []

            for sentence in sentences:

                results.append({
                    '키워드': current_keyword,
                    '연도': current_year,
                    '언론사': media,
                    'URL': url,
                    '추출된_문장': sentence
                })

            return {
                "success": True,
                "url": url,
                "data": results
            }

        with ThreadPoolExecutor(
            max_workers=30
        ) as executor:

            futures = {
                executor.submit(
                    process_url,
                    url
                ): url
                for url in all_unique_urls_to_process
            }

            for future in tqdm(
                as_completed(futures),
                total=len(futures),
                desc=f"'{current_keyword}' ({current_year}) 기사 처리 중"
            ):

                try:

                    result = future.result()

                    if result["success"]:

                        extracted_sentences_data_current_block.extend(
                            result["data"]
                        )

                    else:

                        failed_urls_current_block.append(
                            result["url"]
                        )

                except Exception:

                    failed_urls_current_block.append(
                        futures[future]
                    )

        globally_failed_urls.update(
            failed_urls_current_block
        )

        current_keyword_year_df = pd.DataFrame(
            extracted_sentences_data_current_block
        )

        print(
            f"\n--- '{current_keyword}' 키워드, "
            f"{current_year}년 기사 문장 추출 완료 ---"
        )

        print(
            f"총 {len(current_keyword_year_df)}개의 "
            f"'{current_keyword}' 키워드 문장이 "
            f"{current_year}년 기사에서 추출되었습니다."
        )

        display(
            current_keyword_year_df.head()
        )

        successful_url_fetches = (
            total_urls_to_process
            - len(failed_urls_current_block)
        )

        failed_url_fetches = len(
            failed_urls_current_block
        )

        failed_percentage_of_processed = (
            failed_url_fetches
            / total_urls_to_process
            * 100
        )

        print(
            f"\n--- '{current_keyword}' 키워드, "
            f"{current_year}년 URL 처리 결과 요약 ---"
        )

        print(
            f"전체 엑셀 파일 내 고유 URL 개수: "
            f"{initial_total_urls}개"
        )

        print(
            f"이전에 실패하여 건너뛴 URL: "
            f"{skipped_urls_count}개"
        )

        print(
            f"이번 실행에서 처리 시도한 URL: "
            f"{total_urls_to_process}개"
        )

        print(
            f"기사 내용 추출 성공 URL 개수: "
            f"{successful_url_fetches}개"
        )

        print(
            f"기사 내용 추출 실패 URL 개수: "
            f"{failed_url_fetches}개"
        )

        print(
            f"이번 실행에서의 실패율: "
            f"{failed_percentage_of_processed:.2f}%"
        )

        print(
            f"현재까지 누적된 총 실패 URL 개수: "
            f"{len(globally_failed_urls)}개"
        )

        keyword_specific_extracted_data[
            current_keyword
        ].append(
            current_keyword_year_df
        )

        all_extracted_sentences_for_combined_save.append(
            current_keyword_year_df
        )

        output_keyword_year_path = os.path.join(
            base_path,
            f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx'
        )

        try:

            if not current_keyword_year_df.empty:

                current_keyword_year_df[
                    [
                        '언론사',
                        '추출된_문장',
                        '키워드',
                        '연도',
                        'URL'
                    ]
                ].to_excel(
                    output_keyword_year_path,
                    index=False
                )

                print(
                    f"'{current_keyword}' 키워드, "
                    f"{current_year}년 문장이 "
                    f"저장되었습니다."
                )

            else:

                print(
                    f"'{current_keyword}' 키워드, "
                    f"{current_year}년 추출된 문장이 없습니다."
                )

        except Exception as e:

            print(
                f"엑셀 저장 오류: {e}"
            )

### 4-2-2. '논란' 키워드, 2019년 기사에서 문장 추출 및 파일 저장

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

current_keyword = '논란'
current_year = 2019

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(
    current_keyword,
    current_year,
    all_excel_files
)

if df_keyword_year is None:
    print(
        f"경고: '{current_keyword}' 키워드에 대해 "
        f"{current_year}년도 파일을 찾을 수 없습니다."
    )

else:

    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []

    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = (
            df_keyword_year['URL']
            .dropna()
            .unique()
            .tolist()
        )

    initial_total_urls = len(all_unique_urls_in_excel)

    all_unique_urls_to_process = [
        url
        for url in all_unique_urls_in_excel
        if url not in globally_failed_urls
    ]

    skipped_urls_count = (
        initial_total_urls
        - len(all_unique_urls_to_process)
    )

    if skipped_urls_count > 0:
        print(
            f"이전에 실패하여 건너뛴 URL: "
            f"{skipped_urls_count}개"
        )

    total_urls_to_process = len(
        all_unique_urls_to_process
    )

    if total_urls_to_process == 0:

        if initial_total_urls > 0:
            print(
                f"경고: '{current_keyword}' 키워드, "
                f"{current_year}년 파일의 모든 URL이 "
                f"이전에 실패하여 건너뜁니다."
            )
        else:
            print(
                f"경고: '{current_keyword}' 키워드, "
                f"{current_year}년 파일에 URL이 없습니다."
            )

    else:

        # URL → 언론사 매핑
        url_to_media = (
            df_keyword_year
            .dropna(subset=['URL'])
            .drop_duplicates(subset=['URL'])
            .set_index('URL')['언론사']
            .to_dict()
        )

        def process_url(url):

            article_text = fetch_article_content(url)

            if not article_text:
                return {
                    "success": False,
                    "url": url
                }

            sentences = find_sentences_with_keyword(
                article_text,
                current_keyword
            )

            media = url_to_media.get(
                url,
                'N/A'
            )

            results = []

            for sentence in sentences:

                results.append({
                    '키워드': current_keyword,
                    '연도': current_year,
                    '언론사': media,
                    'URL': url,
                    '추출된_문장': sentence
                })

            return {
                "success": True,
                "url": url,
                "data": results
            }

        with ThreadPoolExecutor(
            max_workers=30
        ) as executor:

            futures = {
                executor.submit(
                    process_url,
                    url
                ): url
                for url in all_unique_urls_to_process
            }

            for future in tqdm(
                as_completed(futures),
                total=len(futures),
                desc=f"'{current_keyword}' ({current_year}) 기사 처리 중"
            ):

                try:

                    result = future.result()

                    if result["success"]:

                        extracted_sentences_data_current_block.extend(
                            result["data"]
                        )

                    else:

                        failed_urls_current_block.append(
                            result["url"]
                        )

                except Exception:

                    failed_urls_current_block.append(
                        futures[future]
                    )

        globally_failed_urls.update(
            failed_urls_current_block
        )

        current_keyword_year_df = pd.DataFrame(
            extracted_sentences_data_current_block
        )

        print(
            f"\n--- '{current_keyword}' 키워드, "
            f"{current_year}년 기사 문장 추출 완료 ---"
        )

        print(
            f"총 {len(current_keyword_year_df)}개의 "
            f"'{current_keyword}' 키워드 문장이 "
            f"{current_year}년 기사에서 추출되었습니다."
        )

        display(
            current_keyword_year_df.head()
        )

        successful_url_fetches = (
            total_urls_to_process
            - len(failed_urls_current_block)
        )

        failed_url_fetches = len(
            failed_urls_current_block
        )

        failed_percentage_of_processed = (
            failed_url_fetches
            / total_urls_to_process
            * 100
        )

        print(
            f"\n--- '{current_keyword}' 키워드, "
            f"{current_year}년 URL 처리 결과 요약 ---"
        )

        print(
            f"전체 엑셀 파일 내 고유 URL 개수: "
            f"{initial_total_urls}개"
        )

        print(
            f"이전에 실패하여 건너뛴 URL: "
            f"{skipped_urls_count}개"
        )

        print(
            f"이번 실행에서 처리 시도한 URL: "
            f"{total_urls_to_process}개"
        )

        print(
            f"기사 내용 추출 성공 URL 개수: "
            f"{successful_url_fetches}개"
        )

        print(
            f"기사 내용 추출 실패 URL 개수: "
            f"{failed_url_fetches}개"
        )

        print(
            f"이번 실행에서의 실패율: "
            f"{failed_percentage_of_processed:.2f}%"
        )

        print(
            f"현재까지 누적된 총 실패 URL 개수: "
            f"{len(globally_failed_urls)}개"
        )

        keyword_specific_extracted_data[
            current_keyword
        ].append(
            current_keyword_year_df
        )

        all_extracted_sentences_for_combined_save.append(
            current_keyword_year_df
        )

        output_keyword_year_path = os.path.join(
            base_path,
            f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx'
        )

        try:

            if not current_keyword_year_df.empty:

                current_keyword_year_df[
                    [
                        '언론사',
                        '추출된_문장',
                        '키워드',
                        '연도',
                        'URL'
                    ]
                ].to_excel(
                    output_keyword_year_path,
                    index=False
                )

                print(
                    f"'{current_keyword}' 키워드, "
                    f"{current_year}년 문장이 "
                    f"저장되었습니다."
                )

            else:

                print(
                    f"'{current_keyword}' 키워드, "
                    f"{current_year}년 추출된 문장이 없습니다."
                )

        except Exception as e:

            print(
                f"엑셀 저장 오류: {e}"
            )

### 4-2-3. '논란' 키워드, 2020년 기사에서 문장 추출 및 파일 저장

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

current_keyword = '논란'
current_year = 2020

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(
    current_keyword,
    current_year,
    all_excel_files
)

if df_keyword_year is None:
    print(
        f"경고: '{current_keyword}' 키워드에 대해 "
        f"{current_year}년도 파일을 찾을 수 없습니다."
    )

else:

    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []

    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = (
            df_keyword_year['URL']
            .dropna()
            .unique()
            .tolist()
        )

    initial_total_urls = len(all_unique_urls_in_excel)

    all_unique_urls_to_process = [
        url
        for url in all_unique_urls_in_excel
        if url not in globally_failed_urls
    ]

    skipped_urls_count = (
        initial_total_urls
        - len(all_unique_urls_to_process)
    )

    if skipped_urls_count > 0:
        print(
            f"이전에 실패하여 건너뛴 URL: "
            f"{skipped_urls_count}개"
        )

    total_urls_to_process = len(
        all_unique_urls_to_process
    )

    if total_urls_to_process == 0:

        if initial_total_urls > 0:
            print(
                f"경고: '{current_keyword}' 키워드, "
                f"{current_year}년 파일의 모든 URL이 "
                f"이전에 실패하여 건너뜁니다."
            )
        else:
            print(
                f"경고: '{current_keyword}' 키워드, "
                f"{current_year}년 파일에 URL이 없습니다."
            )

    else:

        # URL → 언론사 매핑
        url_to_media = (
            df_keyword_year
            .dropna(subset=['URL'])
            .drop_duplicates(subset=['URL'])
            .set_index('URL')['언론사']
            .to_dict()
        )

        def process_url(url):

            article_text = fetch_article_content(url)

            if not article_text:
                return {
                    "success": False,
                    "url": url
                }

            sentences = find_sentences_with_keyword(
                article_text,
                current_keyword
            )

            media = url_to_media.get(
                url,
                'N/A'
            )

            results = []

            for sentence in sentences:

                results.append({
                    '키워드': current_keyword,
                    '연도': current_year,
                    '언론사': media,
                    'URL': url,
                    '추출된_문장': sentence
                })

            return {
                "success": True,
                "url": url,
                "data": results
            }

        with ThreadPoolExecutor(
            max_workers=30
        ) as executor:

            futures = {
                executor.submit(
                    process_url,
                    url
                ): url
                for url in all_unique_urls_to_process
            }

            for future in tqdm(
                as_completed(futures),
                total=len(futures),
                desc=f"'{current_keyword}' ({current_year}) 기사 처리 중"
            ):

                try:

                    result = future.result()

                    if result["success"]:

                        extracted_sentences_data_current_block.extend(
                            result["data"]
                        )

                    else:

                        failed_urls_current_block.append(
                            result["url"]
                        )

                except Exception:

                    failed_urls_current_block.append(
                        futures[future]
                    )

        globally_failed_urls.update(
            failed_urls_current_block
        )

        current_keyword_year_df = pd.DataFrame(
            extracted_sentences_data_current_block
        )

        print(
            f"\n--- '{current_keyword}' 키워드, "
            f"{current_year}년 기사 문장 추출 완료 ---"
        )

        print(
            f"총 {len(current_keyword_year_df)}개의 "
            f"'{current_keyword}' 키워드 문장이 "
            f"{current_year}년 기사에서 추출되었습니다."
        )

        display(
            current_keyword_year_df.head()
        )

        successful_url_fetches = (
            total_urls_to_process
            - len(failed_urls_current_block)
        )

        failed_url_fetches = len(
            failed_urls_current_block
        )

        failed_percentage_of_processed = (
            failed_url_fetches
            / total_urls_to_process
            * 100
        )

        print(
            f"\n--- '{current_keyword}' 키워드, "
            f"{current_year}년 URL 처리 결과 요약 ---"
        )

        print(
            f"전체 엑셀 파일 내 고유 URL 개수: "
            f"{initial_total_urls}개"
        )

        print(
            f"이전에 실패하여 건너뛴 URL: "
            f"{skipped_urls_count}개"
        )

        print(
            f"이번 실행에서 처리 시도한 URL: "
            f"{total_urls_to_process}개"
        )

        print(
            f"기사 내용 추출 성공 URL 개수: "
            f"{successful_url_fetches}개"
        )

        print(
            f"기사 내용 추출 실패 URL 개수: "
            f"{failed_url_fetches}개"
        )

        print(
            f"이번 실행에서의 실패율: "
            f"{failed_percentage_of_processed:.2f}%"
        )

        print(
            f"현재까지 누적된 총 실패 URL 개수: "
            f"{len(globally_failed_urls)}개"
        )

        keyword_specific_extracted_data[
            current_keyword
        ].append(
            current_keyword_year_df
        )

        all_extracted_sentences_for_combined_save.append(
            current_keyword_year_df
        )

        output_keyword_year_path = os.path.join(
            base_path,
            f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx'
        )

        try:

            if not current_keyword_year_df.empty:

                current_keyword_year_df[
                    [
                        '언론사',
                        '추출된_문장',
                        '키워드',
                        '연도',
                        'URL'
                    ]
                ].to_excel(
                    output_keyword_year_path,
                    index=False
                )

                print(
                    f"'{current_keyword}' 키워드, "
                    f"{current_year}년 문장이 "
                    f"저장되었습니다."
                )

            else:

                print(
                    f"'{current_keyword}' 키워드, "
                    f"{current_year}년 추출된 문장이 없습니다."
                )

        except Exception as e:

            print(
                f"엑셀 저장 오류: {e}"
            )

### 4-2-4. '논란' 키워드, 2021년 기사에서 문장 추출 및 파일 저장

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

current_keyword = '논란'
current_year = 2021

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(
    current_keyword,
    current_year,
    all_excel_files
)

if df_keyword_year is None:
    print(
        f"경고: '{current_keyword}' 키워드에 대해 "
        f"{current_year}년도 파일을 찾을 수 없습니다."
    )

else:

    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []

    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = (
            df_keyword_year['URL']
            .dropna()
            .unique()
            .tolist()
        )

    initial_total_urls = len(all_unique_urls_in_excel)

    all_unique_urls_to_process = [
        url
        for url in all_unique_urls_in_excel
        if url not in globally_failed_urls
    ]

    skipped_urls_count = (
        initial_total_urls
        - len(all_unique_urls_to_process)
    )

    if skipped_urls_count > 0:
        print(
            f"이전에 실패하여 건너뛴 URL: "
            f"{skipped_urls_count}개"
        )

    total_urls_to_process = len(
        all_unique_urls_to_process
    )

    if total_urls_to_process == 0:

        if initial_total_urls > 0:
            print(
                f"경고: '{current_keyword}' 키워드, "
                f"{current_year}년 파일의 모든 URL이 "
                f"이전에 실패하여 건너뜁니다."
            )
        else:
            print(
                f"경고: '{current_keyword}' 키워드, "
                f"{current_year}년 파일에 URL이 없습니다."
            )

    else:

        # URL → 언론사 매핑
        url_to_media = (
            df_keyword_year
            .dropna(subset=['URL'])
            .drop_duplicates(subset=['URL'])
            .set_index('URL')['언론사']
            .to_dict()
        )

        def process_url(url):

            article_text = fetch_article_content(url)

            if not article_text:
                return {
                    "success": False,
                    "url": url
                }

            sentences = find_sentences_with_keyword(
                article_text,
                current_keyword
            )

            media = url_to_media.get(
                url,
                'N/A'
            )

            results = []

            for sentence in sentences:

                results.append({
                    '키워드': current_keyword,
                    '연도': current_year,
                    '언론사': media,
                    'URL': url,
                    '추출된_문장': sentence
                })

            return {
                "success": True,
                "url": url,
                "data": results
            }

        with ThreadPoolExecutor(
            max_workers=30
        ) as executor:

            futures = {
                executor.submit(
                    process_url,
                    url
                ): url
                for url in all_unique_urls_to_process
            }

            for future in tqdm(
                as_completed(futures),
                total=len(futures),
                desc=f"'{current_keyword}' ({current_year}) 기사 처리 중"
            ):

                try:

                    result = future.result()

                    if result["success"]:

                        extracted_sentences_data_current_block.extend(
                            result["data"]
                        )

                    else:

                        failed_urls_current_block.append(
                            result["url"]
                        )

                except Exception:

                    failed_urls_current_block.append(
                        futures[future]
                    )

        globally_failed_urls.update(
            failed_urls_current_block
        )

        current_keyword_year_df = pd.DataFrame(
            extracted_sentences_data_current_block
        )

        print(
            f"\n--- '{current_keyword}' 키워드, "
            f"{current_year}년 기사 문장 추출 완료 ---"
        )

        print(
            f"총 {len(current_keyword_year_df)}개의 "
            f"'{current_keyword}' 키워드 문장이 "
            f"{current_year}년 기사에서 추출되었습니다."
        )

        display(
            current_keyword_year_df.head()
        )

        successful_url_fetches = (
            total_urls_to_process
            - len(failed_urls_current_block)
        )

        failed_url_fetches = len(
            failed_urls_current_block
        )

        failed_percentage_of_processed = (
            failed_url_fetches
            / total_urls_to_process
            * 100
        )

        print(
            f"\n--- '{current_keyword}' 키워드, "
            f"{current_year}년 URL 처리 결과 요약 ---"
        )

        print(
            f"전체 엑셀 파일 내 고유 URL 개수: "
            f"{initial_total_urls}개"
        )

        print(
            f"이전에 실패하여 건너뛴 URL: "
            f"{skipped_urls_count}개"
        )

        print(
            f"이번 실행에서 처리 시도한 URL: "
            f"{total_urls_to_process}개"
        )

        print(
            f"기사 내용 추출 성공 URL 개수: "
            f"{successful_url_fetches}개"
        )

        print(
            f"기사 내용 추출 실패 URL 개수: "
            f"{failed_url_fetches}개"
        )

        print(
            f"이번 실행에서의 실패율: "
            f"{failed_percentage_of_processed:.2f}%"
        )

        print(
            f"현재까지 누적된 총 실패 URL 개수: "
            f"{len(globally_failed_urls)}개"
        )

        keyword_specific_extracted_data[
            current_keyword
        ].append(
            current_keyword_year_df
        )

        all_extracted_sentences_for_combined_save.append(
            current_keyword_year_df
        )

        output_keyword_year_path = os.path.join(
            base_path,
            f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx'
        )

        try:

            if not current_keyword_year_df.empty:

                current_keyword_year_df[
                    [
                        '언론사',
                        '추출된_문장',
                        '키워드',
                        '연도',
                        'URL'
                    ]
                ].to_excel(
                    output_keyword_year_path,
                    index=False
                )

                print(
                    f"'{current_keyword}' 키워드, "
                    f"{current_year}년 문장이 "
                    f"저장되었습니다."
                )

            else:

                print(
                    f"'{current_keyword}' 키워드, "
                    f"{current_year}년 추출된 문장이 없습니다."
                )

        except Exception as e:

            print(
                f"엑셀 저장 오류: {e}"
            )

### 4-2-5. '논란' 키워드, 2022년 기사에서 문장 추출 및 파일 저장

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

current_keyword = '논란'
current_year = 2022

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(
    current_keyword,
    current_year,
    all_excel_files
)

if df_keyword_year is None:
    print(
        f"경고: '{current_keyword}' 키워드에 대해 "
        f"{current_year}년도 파일을 찾을 수 없습니다."
    )

else:

    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []

    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = (
            df_keyword_year['URL']
            .dropna()
            .unique()
            .tolist()
        )

    initial_total_urls = len(all_unique_urls_in_excel)

    all_unique_urls_to_process = [
        url
        for url in all_unique_urls_in_excel
        if url not in globally_failed_urls
    ]

    skipped_urls_count = (
        initial_total_urls
        - len(all_unique_urls_to_process)
    )

    if skipped_urls_count > 0:
        print(
            f"이전에 실패하여 건너뛴 URL: "
            f"{skipped_urls_count}개"
        )

    total_urls_to_process = len(
        all_unique_urls_to_process
    )

    if total_urls_to_process == 0:

        if initial_total_urls > 0:
            print(
                f"경고: '{current_keyword}' 키워드, "
                f"{current_year}년 파일의 모든 URL이 "
                f"이전에 실패하여 건너뜁니다."
            )
        else:
            print(
                f"경고: '{current_keyword}' 키워드, "
                f"{current_year}년 파일에 URL이 없습니다."
            )

    else:

        # URL → 언론사 매핑
        url_to_media = (
            df_keyword_year
            .dropna(subset=['URL'])
            .drop_duplicates(subset=['URL'])
            .set_index('URL')['언론사']
            .to_dict()
        )

        def process_url(url):

            article_text = fetch_article_content(url)

            if not article_text:
                return {
                    "success": False,
                    "url": url
                }

            sentences = find_sentences_with_keyword(
                article_text,
                current_keyword
            )

            media = url_to_media.get(
                url,
                'N/A'
            )

            results = []

            for sentence in sentences:

                results.append({
                    '키워드': current_keyword,
                    '연도': current_year,
                    '언론사': media,
                    'URL': url,
                    '추출된_문장': sentence
                })

            return {
                "success": True,
                "url": url,
                "data": results
            }

        with ThreadPoolExecutor(
            max_workers=30
        ) as executor:

            futures = {
                executor.submit(
                    process_url,
                    url
                ): url
                for url in all_unique_urls_to_process
            }

            for future in tqdm(
                as_completed(futures),
                total=len(futures),
                desc=f"'{current_keyword}' ({current_year}) 기사 처리 중"
            ):

                try:

                    result = future.result()

                    if result["success"]:

                        extracted_sentences_data_current_block.extend(
                            result["data"]
                        )

                    else:

                        failed_urls_current_block.append(
                            result["url"]
                        )

                except Exception:

                    failed_urls_current_block.append(
                        futures[future]
                    )

        globally_failed_urls.update(
            failed_urls_current_block
        )

        current_keyword_year_df = pd.DataFrame(
            extracted_sentences_data_current_block
        )

        print(
            f"\n--- '{current_keyword}' 키워드, "
            f"{current_year}년 기사 문장 추출 완료 ---"
        )

        print(
            f"총 {len(current_keyword_year_df)}개의 "
            f"'{current_keyword}' 키워드 문장이 "
            f"{current_year}년 기사에서 추출되었습니다."
        )

        display(
            current_keyword_year_df.head()
        )

        successful_url_fetches = (
            total_urls_to_process
            - len(failed_urls_current_block)
        )

        failed_url_fetches = len(
            failed_urls_current_block
        )

        failed_percentage_of_processed = (
            failed_url_fetches
            / total_urls_to_process
            * 100
        )

        print(
            f"\n--- '{current_keyword}' 키워드, "
            f"{current_year}년 URL 처리 결과 요약 ---"
        )

        print(
            f"전체 엑셀 파일 내 고유 URL 개수: "
            f"{initial_total_urls}개"
        )

        print(
            f"이전에 실패하여 건너뛴 URL: "
            f"{skipped_urls_count}개"
        )

        print(
            f"이번 실행에서 처리 시도한 URL: "
            f"{total_urls_to_process}개"
        )

        print(
            f"기사 내용 추출 성공 URL 개수: "
            f"{successful_url_fetches}개"
        )

        print(
            f"기사 내용 추출 실패 URL 개수: "
            f"{failed_url_fetches}개"
        )

        print(
            f"이번 실행에서의 실패율: "
            f"{failed_percentage_of_processed:.2f}%"
        )

        print(
            f"현재까지 누적된 총 실패 URL 개수: "
            f"{len(globally_failed_urls)}개"
        )

        keyword_specific_extracted_data[
            current_keyword
        ].append(
            current_keyword_year_df
        )

        all_extracted_sentences_for_combined_save.append(
            current_keyword_year_df
        )

        output_keyword_year_path = os.path.join(
            base_path,
            f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx'
        )

        try:

            if not current_keyword_year_df.empty:

                current_keyword_year_df[
                    [
                        '언론사',
                        '추출된_문장',
                        '키워드',
                        '연도',
                        'URL'
                    ]
                ].to_excel(
                    output_keyword_year_path,
                    index=False
                )

                print(
                    f"'{current_keyword}' 키워드, "
                    f"{current_year}년 문장이 "
                    f"저장되었습니다."
                )

            else:

                print(
                    f"'{current_keyword}' 키워드, "
                    f"{current_year}년 추출된 문장이 없습니다."
                )

        except Exception as e:

            print(
                f"엑셀 저장 오류: {e}"
            )

### 4-2-6. '논란' 키워드, 2023년 기사에서 문장 추출 및 파일 저장

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

current_keyword = '논란'
current_year = 2023

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(
    current_keyword,
    current_year,
    all_excel_files
)

if df_keyword_year is None:
    print(
        f"경고: '{current_keyword}' 키워드에 대해 "
        f"{current_year}년도 파일을 찾을 수 없습니다."
    )

else:

    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []

    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = (
            df_keyword_year['URL']
            .dropna()
            .unique()
            .tolist()
        )

    initial_total_urls = len(all_unique_urls_in_excel)

    all_unique_urls_to_process = [
        url
        for url in all_unique_urls_in_excel
        if url not in globally_failed_urls
    ]

    skipped_urls_count = (
        initial_total_urls
        - len(all_unique_urls_to_process)
    )

    if skipped_urls_count > 0:
        print(
            f"이전에 실패하여 건너뛴 URL: "
            f"{skipped_urls_count}개"
        )

    total_urls_to_process = len(
        all_unique_urls_to_process
    )

    if total_urls_to_process == 0:

        if initial_total_urls > 0:
            print(
                f"경고: '{current_keyword}' 키워드, "
                f"{current_year}년 파일의 모든 URL이 "
                f"이전에 실패하여 건너뜁니다."
            )
        else:
            print(
                f"경고: '{current_keyword}' 키워드, "
                f"{current_year}년 파일에 URL이 없습니다."
            )

    else:

        # URL → 언론사 매핑
        url_to_media = (
            df_keyword_year
            .dropna(subset=['URL'])
            .drop_duplicates(subset=['URL'])
            .set_index('URL')['언론사']
            .to_dict()
        )

        def process_url(url):

            article_text = fetch_article_content(url)

            if not article_text:
                return {
                    "success": False,
                    "url": url
                }

            sentences = find_sentences_with_keyword(
                article_text,
                current_keyword
            )

            media = url_to_media.get(
                url,
                'N/A'
            )

            results = []

            for sentence in sentences:

                results.append({
                    '키워드': current_keyword,
                    '연도': current_year,
                    '언론사': media,
                    'URL': url,
                    '추출된_문장': sentence
                })

            return {
                "success": True,
                "url": url,
                "data": results
            }

        with ThreadPoolExecutor(
            max_workers=30
        ) as executor:

            futures = {
                executor.submit(
                    process_url,
                    url
                ): url
                for url in all_unique_urls_to_process
            }

            for future in tqdm(
                as_completed(futures),
                total=len(futures),
                desc=f"'{current_keyword}' ({current_year}) 기사 처리 중"
            ):

                try:

                    result = future.result()

                    if result["success"]:

                        extracted_sentences_data_current_block.extend(
                            result["data"]
                        )

                    else:

                        failed_urls_current_block.append(
                            result["url"]
                        )

                except Exception:

                    failed_urls_current_block.append(
                        futures[future]
                    )

        globally_failed_urls.update(
            failed_urls_current_block
        )

        current_keyword_year_df = pd.DataFrame(
            extracted_sentences_data_current_block
        )

        print(
            f"\n--- '{current_keyword}' 키워드, "
            f"{current_year}년 기사 문장 추출 완료 ---"
        )

        print(
            f"총 {len(current_keyword_year_df)}개의 "
            f"'{current_keyword}' 키워드 문장이 "
            f"{current_year}년 기사에서 추출되었습니다."
        )

        display(
            current_keyword_year_df.head()
        )

        successful_url_fetches = (
            total_urls_to_process
            - len(failed_urls_current_block)
        )

        failed_url_fetches = len(
            failed_urls_current_block
        )

        failed_percentage_of_processed = (
            failed_url_fetches
            / total_urls_to_process
            * 100
        )

        print(
            f"\n--- '{current_keyword}' 키워드, "
            f"{current_year}년 URL 처리 결과 요약 ---"
        )

        print(
            f"전체 엑셀 파일 내 고유 URL 개수: "
            f"{initial_total_urls}개"
        )

        print(
            f"이전에 실패하여 건너뛴 URL: "
            f"{skipped_urls_count}개"
        )

        print(
            f"이번 실행에서 처리 시도한 URL: "
            f"{total_urls_to_process}개"
        )

        print(
            f"기사 내용 추출 성공 URL 개수: "
            f"{successful_url_fetches}개"
        )

        print(
            f"기사 내용 추출 실패 URL 개수: "
            f"{failed_url_fetches}개"
        )

        print(
            f"이번 실행에서의 실패율: "
            f"{failed_percentage_of_processed:.2f}%"
        )

        print(
            f"현재까지 누적된 총 실패 URL 개수: "
            f"{len(globally_failed_urls)}개"
        )

        keyword_specific_extracted_data[
            current_keyword
        ].append(
            current_keyword_year_df
        )

        all_extracted_sentences_for_combined_save.append(
            current_keyword_year_df
        )

        output_keyword_year_path = os.path.join(
            base_path,
            f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx'
        )

        try:

            if not current_keyword_year_df.empty:

                current_keyword_year_df[
                    [
                        '언론사',
                        '추출된_문장',
                        '키워드',
                        '연도',
                        'URL'
                    ]
                ].to_excel(
                    output_keyword_year_path,
                    index=False
                )

                print(
                    f"'{current_keyword}' 키워드, "
                    f"{current_year}년 문장이 "
                    f"저장되었습니다."
                )

            else:

                print(
                    f"'{current_keyword}' 키워드, "
                    f"{current_year}년 추출된 문장이 없습니다."
                )

        except Exception as e:

            print(
                f"엑셀 저장 오류: {e}"
            )

### 4-2-7. '논란' 키워드, 2024년 기사에서 문장 추출 및 파일 저장

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

current_keyword = '논란'
current_year = 2024

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(
    current_keyword,
    current_year,
    all_excel_files
)

if df_keyword_year is None:
    print(
        f"경고: '{current_keyword}' 키워드에 대해 "
        f"{current_year}년도 파일을 찾을 수 없습니다."
    )

else:

    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []

    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = (
            df_keyword_year['URL']
            .dropna()
            .unique()
            .tolist()
        )

    initial_total_urls = len(all_unique_urls_in_excel)

    all_unique_urls_to_process = [
        url
        for url in all_unique_urls_in_excel
        if url not in globally_failed_urls
    ]

    skipped_urls_count = (
        initial_total_urls
        - len(all_unique_urls_to_process)
    )

    if skipped_urls_count > 0:
        print(
            f"이전에 실패하여 건너뛴 URL: "
            f"{skipped_urls_count}개"
        )

    total_urls_to_process = len(
        all_unique_urls_to_process
    )

    if total_urls_to_process == 0:

        if initial_total_urls > 0:
            print(
                f"경고: '{current_keyword}' 키워드, "
                f"{current_year}년 파일의 모든 URL이 "
                f"이전에 실패하여 건너뜁니다."
            )
        else:
            print(
                f"경고: '{current_keyword}' 키워드, "
                f"{current_year}년 파일에 URL이 없습니다."
            )

    else:

        # URL → 언론사 매핑
        url_to_media = (
            df_keyword_year
            .dropna(subset=['URL'])
            .drop_duplicates(subset=['URL'])
            .set_index('URL')['언론사']
            .to_dict()
        )

        def process_url(url):

            article_text = fetch_article_content(url)

            if not article_text:
                return {
                    "success": False,
                    "url": url
                }

            sentences = find_sentences_with_keyword(
                article_text,
                current_keyword
            )

            media = url_to_media.get(
                url,
                'N/A'
            )

            results = []

            for sentence in sentences:

                results.append({
                    '키워드': current_keyword,
                    '연도': current_year,
                    '언론사': media,
                    'URL': url,
                    '추출된_문장': sentence
                })

            return {
                "success": True,
                "url": url,
                "data": results
            }

        with ThreadPoolExecutor(
            max_workers=30
        ) as executor:

            futures = {
                executor.submit(
                    process_url,
                    url
                ): url
                for url in all_unique_urls_to_process
            }

            for future in tqdm(
                as_completed(futures),
                total=len(futures),
                desc=f"'{current_keyword}' ({current_year}) 기사 처리 중"
            ):

                try:

                    result = future.result()

                    if result["success"]:

                        extracted_sentences_data_current_block.extend(
                            result["data"]
                        )

                    else:

                        failed_urls_current_block.append(
                            result["url"]
                        )

                except Exception:

                    failed_urls_current_block.append(
                        futures[future]
                    )

        globally_failed_urls.update(
            failed_urls_current_block
        )

        current_keyword_year_df = pd.DataFrame(
            extracted_sentences_data_current_block
        )

        print(
            f"\n--- '{current_keyword}' 키워드, "
            f"{current_year}년 기사 문장 추출 완료 ---"
        )

        print(
            f"총 {len(current_keyword_year_df)}개의 "
            f"'{current_keyword}' 키워드 문장이 "
            f"{current_year}년 기사에서 추출되었습니다."
        )

        display(
            current_keyword_year_df.head()
        )

        successful_url_fetches = (
            total_urls_to_process
            - len(failed_urls_current_block)
        )

        failed_url_fetches = len(
            failed_urls_current_block
        )

        failed_percentage_of_processed = (
            failed_url_fetches
            / total_urls_to_process
            * 100
        )

        print(
            f"\n--- '{current_keyword}' 키워드, "
            f"{current_year}년 URL 처리 결과 요약 ---"
        )

        print(
            f"전체 엑셀 파일 내 고유 URL 개수: "
            f"{initial_total_urls}개"
        )

        print(
            f"이전에 실패하여 건너뛴 URL: "
            f"{skipped_urls_count}개"
        )

        print(
            f"이번 실행에서 처리 시도한 URL: "
            f"{total_urls_to_process}개"
        )

        print(
            f"기사 내용 추출 성공 URL 개수: "
            f"{successful_url_fetches}개"
        )

        print(
            f"기사 내용 추출 실패 URL 개수: "
            f"{failed_url_fetches}개"
        )

        print(
            f"이번 실행에서의 실패율: "
            f"{failed_percentage_of_processed:.2f}%"
        )

        print(
            f"현재까지 누적된 총 실패 URL 개수: "
            f"{len(globally_failed_urls)}개"
        )

        keyword_specific_extracted_data[
            current_keyword
        ].append(
            current_keyword_year_df
        )

        all_extracted_sentences_for_combined_save.append(
            current_keyword_year_df
        )

        output_keyword_year_path = os.path.join(
            base_path,
            f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx'
        )

        try:

            if not current_keyword_year_df.empty:

                current_keyword_year_df[
                    [
                        '언론사',
                        '추출된_문장',
                        '키워드',
                        '연도',
                        'URL'
                    ]
                ].to_excel(
                    output_keyword_year_path,
                    index=False
                )

                print(
                    f"'{current_keyword}' 키워드, "
                    f"{current_year}년 문장이 "
                    f"저장되었습니다."
                )

            else:

                print(
                    f"'{current_keyword}' 키워드, "
                    f"{current_year}년 추출된 문장이 없습니다."
                )

        except Exception as e:

            print(
                f"엑셀 저장 오류: {e}"
            )

### 4-2-8. '논란' 키워드, 2025년 기사에서 문장 추출 및 파일 저장

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

current_keyword = '논란'
current_year = 2025

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(
    current_keyword,
    current_year,
    all_excel_files
)

if df_keyword_year is None:
    print(
        f"경고: '{current_keyword}' 키워드에 대해 "
        f"{current_year}년도 파일을 찾을 수 없습니다."
    )

else:

    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []

    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = (
            df_keyword_year['URL']
            .dropna()
            .unique()
            .tolist()
        )

    initial_total_urls = len(all_unique_urls_in_excel)

    all_unique_urls_to_process = [
        url
        for url in all_unique_urls_in_excel
        if url not in globally_failed_urls
    ]

    skipped_urls_count = (
        initial_total_urls
        - len(all_unique_urls_to_process)
    )

    if skipped_urls_count > 0:
        print(
            f"이전에 실패하여 건너뛴 URL: "
            f"{skipped_urls_count}개"
        )

    total_urls_to_process = len(
        all_unique_urls_to_process
    )

    if total_urls_to_process == 0:

        if initial_total_urls > 0:
            print(
                f"경고: '{current_keyword}' 키워드, "
                f"{current_year}년 파일의 모든 URL이 "
                f"이전에 실패하여 건너뜁니다."
            )
        else:
            print(
                f"경고: '{current_keyword}' 키워드, "
                f"{current_year}년 파일에 URL이 없습니다."
            )

    else:

        # URL → 언론사 매핑
        url_to_media = (
            df_keyword_year
            .dropna(subset=['URL'])
            .drop_duplicates(subset=['URL'])
            .set_index('URL')['언론사']
            .to_dict()
        )

        def process_url(url):

            article_text = fetch_article_content(url)

            if not article_text:
                return {
                    "success": False,
                    "url": url
                }

            sentences = find_sentences_with_keyword(
                article_text,
                current_keyword
            )

            media = url_to_media.get(
                url,
                'N/A'
            )

            results = []

            for sentence in sentences:

                results.append({
                    '키워드': current_keyword,
                    '연도': current_year,
                    '언론사': media,
                    'URL': url,
                    '추출된_문장': sentence
                })

            return {
                "success": True,
                "url": url,
                "data": results
            }

        with ThreadPoolExecutor(
            max_workers=30
        ) as executor:

            futures = {
                executor.submit(
                    process_url,
                    url
                ): url
                for url in all_unique_urls_to_process
            }

            for future in tqdm(
                as_completed(futures),
                total=len(futures),
                desc=f"'{current_keyword}' ({current_year}) 기사 처리 중"
            ):

                try:

                    result = future.result()

                    if result["success"]:

                        extracted_sentences_data_current_block.extend(
                            result["data"]
                        )

                    else:

                        failed_urls_current_block.append(
                            result["url"]
                        )

                except Exception:

                    failed_urls_current_block.append(
                        futures[future]
                    )

        globally_failed_urls.update(
            failed_urls_current_block
        )

        current_keyword_year_df = pd.DataFrame(
            extracted_sentences_data_current_block
        )

        print(
            f"\n--- '{current_keyword}' 키워드, "
            f"{current_year}년 기사 문장 추출 완료 ---"
        )

        print(
            f"총 {len(current_keyword_year_df)}개의 "
            f"'{current_keyword}' 키워드 문장이 "
            f"{current_year}년 기사에서 추출되었습니다."
        )

        display(
            current_keyword_year_df.head()
        )

        successful_url_fetches = (
            total_urls_to_process
            - len(failed_urls_current_block)
        )

        failed_url_fetches = len(
            failed_urls_current_block
        )

        failed_percentage_of_processed = (
            failed_url_fetches
            / total_urls_to_process
            * 100
        )

        print(
            f"\n--- '{current_keyword}' 키워드, "
            f"{current_year}년 URL 처리 결과 요약 ---"
        )

        print(
            f"전체 엑셀 파일 내 고유 URL 개수: "
            f"{initial_total_urls}개"
        )

        print(
            f"이전에 실패하여 건너뛴 URL: "
            f"{skipped_urls_count}개"
        )

        print(
            f"이번 실행에서 처리 시도한 URL: "
            f"{total_urls_to_process}개"
        )

        print(
            f"기사 내용 추출 성공 URL 개수: "
            f"{successful_url_fetches}개"
        )

        print(
            f"기사 내용 추출 실패 URL 개수: "
            f"{failed_url_fetches}개"
        )

        print(
            f"이번 실행에서의 실패율: "
            f"{failed_percentage_of_processed:.2f}%"
        )

        print(
            f"현재까지 누적된 총 실패 URL 개수: "
            f"{len(globally_failed_urls)}개"
        )

        keyword_specific_extracted_data[
            current_keyword
        ].append(
            current_keyword_year_df
        )

        all_extracted_sentences_for_combined_save.append(
            current_keyword_year_df
        )

        output_keyword_year_path = os.path.join(
            base_path,
            f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx'
        )

        try:

            if not current_keyword_year_df.empty:

                current_keyword_year_df[
                    [
                        '언론사',
                        '추출된_문장',
                        '키워드',
                        '연도',
                        'URL'
                    ]
                ].to_excel(
                    output_keyword_year_path,
                    index=False
                )

                print(
                    f"'{current_keyword}' 키워드, "
                    f"{current_year}년 문장이 "
                    f"저장되었습니다."
                )

            else:

                print(
                    f"'{current_keyword}' 키워드, "
                    f"{current_year}년 추출된 문장이 없습니다."
                )

        except Exception as e:

            print(
                f"엑셀 저장 오류: {e}"
            )

### 4-3-1. '이슈' 키워드, 2018년 기사에서 문장 추출 및 파일 저장

In [ ]:
current_keyword = '이슈'
current_year = 2018

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(current_keyword, current_year, all_excel_files)

if df_keyword_year is None:
    print(f"경고: '{current_keyword}' 키워드에 대해 {current_year}년도 파일을 찾을 수 없습니다.")
else:
    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []
    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = df_keyword_year['URL'].dropna().unique().tolist()

    total_urls_in_excel = len(all_unique_urls_in_excel)

    if total_urls_in_excel == 0:
        print(f"경고: '{current_keyword}' 키워드, {current_year}년 파일에 'URL' 컬럼이 없거나 모든 URL이 비어있습니다. 건너뜨니다.")
    else:
        with tqdm(total=total_urls_in_excel, desc=f"'{current_keyword}' ({current_year}) 기사 처리 중") as pbar:
            for url in all_unique_urls_in_excel:
                import io
                import contextlib
                with contextlib.redirect_stdout(io.StringIO()):
                    article_text = fetch_article_content(url)

                if article_text:
                    sentences = find_sentences_with_keyword(article_text, current_keyword)
                    # '언론사'는 해당 URL이 원본 데이터에 여러 번 나타날 경우 불일치할 수 있으므로, 해당 URL과 일치하는 첫 번째 언론사를 사용합니다.
                    media = 'N/A'
                    matching_rows = df_keyword_year[df_keyword_year['URL'] == url]
                    if not matching_rows.empty and '언론사' in matching_rows.columns:
                        first_media = matching_rows['언론사'].dropna().iloc[0] if not matching_rows['언론사'].dropna().empty else 'N/A'
                        media = first_media

                    if sentences:
                        for sentence in sentences:
                            extracted_sentences_data_current_block.append({
                                '키워드': current_keyword,
                                '연도': current_year,
                                '언론사': media,
                                'URL': url,
                                '추출된_문장': sentence
                            })
                else:
                    failed_urls_current_block.append(url) # Store failed URL
                pbar.update(1)
                time.sleep(0.5)

        current_keyword_year_df = pd.DataFrame(extracted_sentences_data_current_block)

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 완료 ---")
        print(f"총 {len(current_keyword_year_df)}개의 '{current_keyword}' 키워드 문장이 {current_year}년 기사에서 추출되었습니다.")
        print("추출된 문장 DataFrame 미리보기:")
        display(current_keyword_year_df.head())

        # URL 처리 결과 요약
        successful_url_fetches = total_urls_in_excel - len(failed_urls_current_block)
        failed_url_fetches = len(failed_urls_current_block)
        failed_percentage = (failed_url_fetches / total_urls_in_excel * 100) if total_urls_in_excel > 0 else 0

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 URL 처리 결과 요약 ---")
        print(f"전체 엑셀 파일 내 고유 URL 개수: {total_urls_in_excel}개")
        print(f"기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): {successful_url_fetches}개")
        print(f"기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): {failed_url_fetches}개")
        print(f"실패율: {failed_percentage:.2f}%")


        # if failed_urls_current_block:
        #     print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL 목록 ({len(failed_urls_current_block)}개) ---")
        #     for failed_url in failed_urls_current_block:
        #         print(failed_url)
        # else:
        #     print(f"\n'{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL은 없습니다.")

        # 키워드별 추출된 데이터 리스트에 추가 (나중에 키워드별 통합 파일을 위해)
        keyword_specific_extracted_data[current_keyword].append(current_keyword_year_df)
        # 모든 키워드의 통합 저장을 위한 리스트에 추가
        all_extracted_sentences_for_combined_save.append(current_keyword_year_df)

        # 현재 키워드-연도 데이터를 별도 엑셀 파일로 저장
        output_keyword_year_path = os.path.join(base_path, f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx')
        try:
            if not current_keyword_year_df.empty:
                current_keyword_year_df[['언론사', '추출된_문장', '키워드', '연도', 'URL']].to_excel(output_keyword_year_path, index=False)
                print(f"'{current_keyword}' 키워드, {current_year}년 문장이 '{output_keyword_year_path}'에 성공적으로 저장되었습니다.")
            else:
                print(f"'{current_keyword}' 키워드, {current_year}년 추출된 문장이 없어 파일을 저장하지 않습니다.")
        except Exception as e:
            print(f"'{current_keyword}' 키워드, {current_year}년 엑셀 파일 저장 중 오류 발생: {e}")

### 4-3-2. '이슈' 키워드, 2019년 기사에서 문장 추출 및 파일 저장

In [ ]:
current_keyword = '이슈'
current_year = 2019

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(current_keyword, current_year, all_excel_files)

if df_keyword_year is None:
    print(f"경고: '{current_keyword}' 키워드에 대해 {current_year}년도 파일을 찾을 수 없습니다.")
else:
    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []
    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = df_keyword_year['URL'].dropna().unique().tolist()

    total_urls_in_excel = len(all_unique_urls_in_excel)

    if total_urls_in_excel == 0:
        print(f"경고: '{current_keyword}' 키워드, {current_year}년 파일에 'URL' 컬럼이 없거나 모든 URL이 비어있습니다. 건너뜨니다.")
    else:
        with tqdm(total=total_urls_in_excel, desc=f"'{current_keyword}' ({current_year}) 기사 처리 중") as pbar:
            for url in all_unique_urls_in_excel:
                import io
                import contextlib
                with contextlib.redirect_stdout(io.StringIO()):
                    article_text = fetch_article_content(url)

                if article_text:
                    sentences = find_sentences_with_keyword(article_text, current_keyword)
                    # '언론사'는 해당 URL이 원본 데이터에 여러 번 나타날 경우 불일치할 수 있으므로, 해당 URL과 일치하는 첫 번째 언론사를 사용합니다.
                    media = 'N/A'
                    matching_rows = df_keyword_year[df_keyword_year['URL'] == url]
                    if not matching_rows.empty and '언론사' in matching_rows.columns:
                        first_media = matching_rows['언론사'].dropna().iloc[0] if not matching_rows['언론사'].dropna().empty else 'N/A'
                        media = first_media

                    if sentences:
                        for sentence in sentences:
                            extracted_sentences_data_current_block.append({
                                '키워드': current_keyword,
                                '연도': current_year,
                                '언론사': media,
                                'URL': url,
                                '추출된_문장': sentence
                            })
                else:
                    failed_urls_current_block.append(url) # Store failed URL
                pbar.update(1)
                time.sleep(0.5)

        current_keyword_year_df = pd.DataFrame(extracted_sentences_data_current_block)

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 완료 ---")
        print(f"총 {len(current_keyword_year_df)}개의 '{current_keyword}' 키워드 문장이 {current_year}년 기사에서 추출되었습니다.")
        print("추출된 문장 DataFrame 미리보기:")
        display(current_keyword_year_df.head())

        # URL 처리 결과 요약
        successful_url_fetches = total_urls_in_excel - len(failed_urls_current_block)
        failed_url_fetches = len(failed_urls_current_block)
        failed_percentage = (failed_url_fetches / total_urls_in_excel * 100) if total_urls_in_excel > 0 else 0

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 URL 처리 결과 요약 ---")
        print(f"전체 엑셀 파일 내 고유 URL 개수: {total_urls_in_excel}개")
        print(f"기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): {successful_url_fetches}개")
        print(f"기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): {failed_url_fetches}개")
        print(f"실패율: {failed_percentage:.2f}%")


        # if failed_urls_current_block:
        #     print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL 목록 ({len(failed_urls_current_block)}개) ---")
        #     for failed_url in failed_urls_current_block:
        #         print(failed_url)
        # else:
        #     print(f"\n'{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL은 없습니다.")

        # 키워드별 추출된 데이터 리스트에 추가 (나중에 키워드별 통합 파일을 위해)
        keyword_specific_extracted_data[current_keyword].append(current_keyword_year_df)
        # 모든 키워드의 통합 저장을 위한 리스트에 추가
        all_extracted_sentences_for_combined_save.append(current_keyword_year_df)

        # 현재 키워드-연도 데이터를 별도 엑셀 파일로 저장
        output_keyword_year_path = os.path.join(base_path, f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx')
        try:
            if not current_keyword_year_df.empty:
                current_keyword_year_df[['언론사', '추출된_문장', '키워드', '연도', 'URL']].to_excel(output_keyword_year_path, index=False)
                print(f"'{current_keyword}' 키워드, {current_year}년 문장이 '{output_keyword_year_path}'에 성공적으로 저장되었습니다.")
            else:
                print(f"'{current_keyword}' 키워드, {current_year}년 추출된 문장이 없어 파일을 저장하지 않습니다.")
        except Exception as e:
            print(f"'{current_keyword}' 키워드, {current_year}년 엑셀 파일 저장 중 오류 발생: {e}")

### 4-3-3. '이슈' 키워드, 2020년 기사에서 문장 추출 및 파일 저장

In [ ]:
current_keyword = '이슈'
current_year = 2020

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(current_keyword, current_year, all_excel_files)

if df_keyword_year is None:
    print(f"경고: '{current_keyword}' 키워드에 대해 {current_year}년도 파일을 찾을 수 없습니다.")
else:
    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []
    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = df_keyword_year['URL'].dropna().unique().tolist()

    total_urls_in_excel = len(all_unique_urls_in_excel)

    if total_urls_in_excel == 0:
        print(f"경고: '{current_keyword}' 키워드, {current_year}년 파일에 'URL' 컬럼이 없거나 모든 URL이 비어있습니다. 건너뜨니다.")
    else:
        with tqdm(total=total_urls_in_excel, desc=f"'{current_keyword}' ({current_year}) 기사 처리 중") as pbar:
            for url in all_unique_urls_in_excel:
                import io
                import contextlib
                with contextlib.redirect_stdout(io.StringIO()):
                    article_text = fetch_article_content(url)

                if article_text:
                    sentences = find_sentences_with_keyword(article_text, current_keyword)
                    # '언론사'는 해당 URL이 원본 데이터에 여러 번 나타날 경우 불일치할 수 있으므로, 해당 URL과 일치하는 첫 번째 언론사를 사용합니다.
                    media = 'N/A'
                    matching_rows = df_keyword_year[df_keyword_year['URL'] == url]
                    if not matching_rows.empty and '언론사' in matching_rows.columns:
                        first_media = matching_rows['언론사'].dropna().iloc[0] if not matching_rows['언론사'].dropna().empty else 'N/A'
                        media = first_media

                    if sentences:
                        for sentence in sentences:
                            extracted_sentences_data_current_block.append({
                                '키워드': current_keyword,
                                '연도': current_year,
                                '언론사': media,
                                'URL': url,
                                '추출된_문장': sentence
                            })
                else:
                    failed_urls_current_block.append(url) # Store failed URL
                pbar.update(1)
                time.sleep(0.5)

        current_keyword_year_df = pd.DataFrame(extracted_sentences_data_current_block)

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 완료 ---")
        print(f"총 {len(current_keyword_year_df)}개의 '{current_keyword}' 키워드 문장이 {current_year}년 기사에서 추출되었습니다.")
        print("추출된 문장 DataFrame 미리보기:")
        display(current_keyword_year_df.head())

        # URL 처리 결과 요약
        successful_url_fetches = total_urls_in_excel - len(failed_urls_current_block)
        failed_url_fetches = len(failed_urls_current_block)
        failed_percentage = (failed_url_fetches / total_urls_in_excel * 100) if total_urls_in_excel > 0 else 0

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 URL 처리 결과 요약 ---")
        print(f"전체 엑셀 파일 내 고유 URL 개수: {total_urls_in_excel}개")
        print(f"기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): {successful_url_fetches}개")
        print(f"기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): {failed_url_fetches}개")
        print(f"실패율: {failed_percentage:.2f}%")


        # if failed_urls_current_block:
        #     print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL 목록 ({len(failed_urls_current_block)}개) ---")
        #     for failed_url in failed_urls_current_block:
        #         print(failed_url)
        # else:
        #     print(f"\n'{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL은 없습니다.")

        # 키워드별 추출된 데이터 리스트에 추가 (나중에 키워드별 통합 파일을 위해)
        keyword_specific_extracted_data[current_keyword].append(current_keyword_year_df)
        # 모든 키워드의 통합 저장을 위한 리스트에 추가
        all_extracted_sentences_for_combined_save.append(current_keyword_year_df)

        # 현재 키워드-연도 데이터를 별도 엑셀 파일로 저장
        output_keyword_year_path = os.path.join(base_path, f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx')
        try:
            if not current_keyword_year_df.empty:
                current_keyword_year_df[['언론사', '추출된_문장', '키워드', '연도', 'URL']].to_excel(output_keyword_year_path, index=False)
                print(f"'{current_keyword}' 키워드, {current_year}년 문장이 '{output_keyword_year_path}'에 성공적으로 저장되었습니다.")
            else:
                print(f"'{current_keyword}' 키워드, {current_year}년 추출된 문장이 없어 파일을 저장하지 않습니다.")
        except Exception as e:
            print(f"'{current_keyword}' 키워드, {current_year}년 엑셀 파일 저장 중 오류 발생: {e}")

### 4-3-4. '이슈' 키워드, 2021년 기사에서 문장 추출 및 파일 저장

In [ ]:
current_keyword = '이슈'
current_year = 2021

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(current_keyword, current_year, all_excel_files)

if df_keyword_year is None:
    print(f"경고: '{current_keyword}' 키워드에 대해 {current_year}년도 파일을 찾을 수 없습니다.")
else:
    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []
    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = df_keyword_year['URL'].dropna().unique().tolist()

    total_urls_in_excel = len(all_unique_urls_in_excel)

    if total_urls_in_excel == 0:
        print(f"경고: '{current_keyword}' 키워드, {current_year}년 파일에 'URL' 컬럼이 없거나 모든 URL이 비어있습니다. 건너뜨니다.")
    else:
        with tqdm(total=total_urls_in_excel, desc=f"'{current_keyword}' ({current_year}) 기사 처리 중") as pbar:
            for url in all_unique_urls_in_excel:
                import io
                import contextlib
                with contextlib.redirect_stdout(io.StringIO()):
                    article_text = fetch_article_content(url)

                if article_text:
                    sentences = find_sentences_with_keyword(article_text, current_keyword)
                    # '언론사'는 해당 URL이 원본 데이터에 여러 번 나타날 경우 불일치할 수 있으므로, 해당 URL과 일치하는 첫 번째 언론사를 사용합니다.
                    media = 'N/A'
                    matching_rows = df_keyword_year[df_keyword_year['URL'] == url]
                    if not matching_rows.empty and '언론사' in matching_rows.columns:
                        first_media = matching_rows['언론사'].dropna().iloc[0] if not matching_rows['언론사'].dropna().empty else 'N/A'
                        media = first_media

                    if sentences:
                        for sentence in sentences:
                            extracted_sentences_data_current_block.append({
                                '키워드': current_keyword,
                                '연도': current_year,
                                '언론사': media,
                                'URL': url,
                                '추출된_문장': sentence
                            })
                else:
                    failed_urls_current_block.append(url) # Store failed URL
                pbar.update(1)
                time.sleep(0.5)

        current_keyword_year_df = pd.DataFrame(extracted_sentences_data_current_block)

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 완료 ---")
        print(f"총 {len(current_keyword_year_df)}개의 '{current_keyword}' 키워드 문장이 {current_year}년 기사에서 추출되었습니다.")
        print("추출된 문장 DataFrame 미리보기:")
        display(current_keyword_year_df.head())

        # URL 처리 결과 요약
        successful_url_fetches = total_urls_in_excel - len(failed_urls_current_block)
        failed_url_fetches = len(failed_urls_current_block)
        failed_percentage = (failed_url_fetches / total_urls_in_excel * 100) if total_urls_in_excel > 0 else 0

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 URL 처리 결과 요약 ---")
        print(f"전체 엑셀 파일 내 고유 URL 개수: {total_urls_in_excel}개")
        print(f"기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): {successful_url_fetches}개")
        print(f"기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): {failed_url_fetches}개")
        print(f"실패율: {failed_percentage:.2f}%")


        # if failed_urls_current_block:
        #     print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL 목록 ({len(failed_urls_current_block)}개) ---")
        #     for failed_url in failed_urls_current_block:
        #         print(failed_url)
        # else:
        #     print(f"\n'{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL은 없습니다.")

        # 키워드별 추출된 데이터 리스트에 추가 (나중에 키워드별 통합 파일을 위해)
        keyword_specific_extracted_data[current_keyword].append(current_keyword_year_df)
        # 모든 키워드의 통합 저장을 위한 리스트에 추가
        all_extracted_sentences_for_combined_save.append(current_keyword_year_df)

        # 현재 키워드-연도 데이터를 별도 엑셀 파일로 저장
        output_keyword_year_path = os.path.join(base_path, f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx')
        try:
            if not current_keyword_year_df.empty:
                current_keyword_year_df[['언론사', '추출된_문장', '키워드', '연도', 'URL']].to_excel(output_keyword_year_path, index=False)
                print(f"'{current_keyword}' 키워드, {current_year}년 문장이 '{output_keyword_year_path}'에 성공적으로 저장되었습니다.")
            else:
                print(f"'{current_keyword}' 키워드, {current_year}년 추출된 문장이 없어 파일을 저장하지 않습니다.")
        except Exception as e:
            print(f"'{current_keyword}' 키워드, {current_year}년 엑셀 파일 저장 중 오류 발생: {e}")

### 4-3-5. '이슈' 키워드, 2022년 기사에서 문장 추출 및 파일 저장

In [ ]:
current_keyword = '이슈'
current_year = 2022

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(current_keyword, current_year, all_excel_files)

if df_keyword_year is None:
    print(f"경고: '{current_keyword}' 키워드에 대해 {current_year}년도 파일을 찾을 수 없습니다.")
else:
    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []
    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = df_keyword_year['URL'].dropna().unique().tolist()

    total_urls_in_excel = len(all_unique_urls_in_excel)

    if total_urls_in_excel == 0:
        print(f"경고: '{current_keyword}' 키워드, {current_year}년 파일에 'URL' 컬럼이 없거나 모든 URL이 비어있습니다. 건너뜨니다.")
    else:
        with tqdm(total=total_urls_in_excel, desc=f"'{current_keyword}' ({current_year}) 기사 처리 중") as pbar:
            for url in all_unique_urls_in_excel:
                import io
                import contextlib
                with contextlib.redirect_stdout(io.StringIO()):
                    article_text = fetch_article_content(url)

                if article_text:
                    sentences = find_sentences_with_keyword(article_text, current_keyword)
                    # '언론사'는 해당 URL이 원본 데이터에 여러 번 나타날 경우 불일치할 수 있으므로, 해당 URL과 일치하는 첫 번째 언론사를 사용합니다.
                    media = 'N/A'
                    matching_rows = df_keyword_year[df_keyword_year['URL'] == url]
                    if not matching_rows.empty and '언론사' in matching_rows.columns:
                        first_media = matching_rows['언론사'].dropna().iloc[0] if not matching_rows['언론사'].dropna().empty else 'N/A'
                        media = first_media

                    if sentences:
                        for sentence in sentences:
                            extracted_sentences_data_current_block.append({
                                '키워드': current_keyword,
                                '연도': current_year,
                                '언론사': media,
                                'URL': url,
                                '추출된_문장': sentence
                            })
                else:
                    failed_urls_current_block.append(url) # Store failed URL
                pbar.update(1)
                time.sleep(0.5)

        current_keyword_year_df = pd.DataFrame(extracted_sentences_data_current_block)

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 완료 ---")
        print(f"총 {len(current_keyword_year_df)}개의 '{current_keyword}' 키워드 문장이 {current_year}년 기사에서 추출되었습니다.")
        print("추출된 문장 DataFrame 미리보기:")
        display(current_keyword_year_df.head())

        # URL 처리 결과 요약
        successful_url_fetches = total_urls_in_excel - len(failed_urls_current_block)
        failed_url_fetches = len(failed_urls_current_block)
        failed_percentage = (failed_url_fetches / total_urls_in_excel * 100) if total_urls_in_excel > 0 else 0

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 URL 처리 결과 요약 ---")
        print(f"전체 엑셀 파일 내 고유 URL 개수: {total_urls_in_excel}개")
        print(f"기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): {successful_url_fetches}개")
        print(f"기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): {failed_url_fetches}개")
        print(f"실패율: {failed_percentage:.2f}%")


        # if failed_urls_current_block:
        #     print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL 목록 ({len(failed_urls_current_block)}개) ---")
        #     for failed_url in failed_urls_current_block:
        #         print(failed_url)
        # else:
        #     print(f"\n'{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL은 없습니다.")

        # 키워드별 추출된 데이터 리스트에 추가 (나중에 키워드별 통합 파일을 위해)
        keyword_specific_extracted_data[current_keyword].append(current_keyword_year_df)
        # 모든 키워드의 통합 저장을 위한 리스트에 추가
        all_extracted_sentences_for_combined_save.append(current_keyword_year_df)

        # 현재 키워드-연도 데이터를 별도 엑셀 파일로 저장
        output_keyword_year_path = os.path.join(base_path, f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx')
        try:
            if not current_keyword_year_df.empty:
                current_keyword_year_df[['언론사', '추출된_문장', '키워드', '연도', 'URL']].to_excel(output_keyword_year_path, index=False)
                print(f"'{current_keyword}' 키워드, {current_year}년 문장이 '{output_keyword_year_path}'에 성공적으로 저장되었습니다.")
            else:
                print(f"'{current_keyword}' 키워드, {current_year}년 추출된 문장이 없어 파일을 저장하지 않습니다.")
        except Exception as e:
            print(f"'{current_keyword}' 키워드, {current_year}년 엑셀 파일 저장 중 오류 발생: {e}")

### 4-3-6. '이슈' 키워드, 2023년 기사에서 문장 추출 및 파일 저장

In [ ]:
current_keyword = '이슈'
current_year = 2023

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(current_keyword, current_year, all_excel_files)

if df_keyword_year is None:
    print(f"경고: '{current_keyword}' 키워드에 대해 {current_year}년도 파일을 찾을 수 없습니다.")
else:
    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []
    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = df_keyword_year['URL'].dropna().unique().tolist()

    total_urls_in_excel = len(all_unique_urls_in_excel)

    if total_urls_in_excel == 0:
        print(f"경고: '{current_keyword}' 키워드, {current_year}년 파일에 'URL' 컬럼이 없거나 모든 URL이 비어있습니다. 건너뜨니다.")
    else:
        with tqdm(total=total_urls_in_excel, desc=f"'{current_keyword}' ({current_year}) 기사 처리 중") as pbar:
            for url in all_unique_urls_in_excel:
                import io
                import contextlib
                with contextlib.redirect_stdout(io.StringIO()):
                    article_text = fetch_article_content(url)

                if article_text:
                    sentences = find_sentences_with_keyword(article_text, current_keyword)
                    # '언론사'는 해당 URL이 원본 데이터에 여러 번 나타날 경우 불일치할 수 있으므로, 해당 URL과 일치하는 첫 번째 언론사를 사용합니다.
                    media = 'N/A'
                    matching_rows = df_keyword_year[df_keyword_year['URL'] == url]
                    if not matching_rows.empty and '언론사' in matching_rows.columns:
                        first_media = matching_rows['언론사'].dropna().iloc[0] if not matching_rows['언론사'].dropna().empty else 'N/A'
                        media = first_media

                    if sentences:
                        for sentence in sentences:
                            extracted_sentences_data_current_block.append({
                                '키워드': current_keyword,
                                '연도': current_year,
                                '언론사': media,
                                'URL': url,
                                '추출된_문장': sentence
                            })
                else:
                    failed_urls_current_block.append(url) # Store failed URL
                pbar.update(1)
                time.sleep(0.5)

        current_keyword_year_df = pd.DataFrame(extracted_sentences_data_current_block)

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 완료 ---")
        print(f"총 {len(current_keyword_year_df)}개의 '{current_keyword}' 키워드 문장이 {current_year}년 기사에서 추출되었습니다.")
        print("추출된 문장 DataFrame 미리보기:")
        display(current_keyword_year_df.head())

        # URL 처리 결과 요약
        successful_url_fetches = total_urls_in_excel - len(failed_urls_current_block)
        failed_url_fetches = len(failed_urls_current_block)
        failed_percentage = (failed_url_fetches / total_urls_in_excel * 100) if total_urls_in_excel > 0 else 0

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 URL 처리 결과 요약 ---")
        print(f"전체 엑셀 파일 내 고유 URL 개수: {total_urls_in_excel}개")
        print(f"기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): {successful_url_fetches}개")
        print(f"기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): {failed_url_fetches}개")
        print(f"실패율: {failed_percentage:.2f}%")


        # if failed_urls_current_block:
        #     print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL 목록 ({len(failed_urls_current_block)}개) ---")
        #     for failed_url in failed_urls_current_block:
        #         print(failed_url)
        # else:
        #     print(f"\n'{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL은 없습니다.")

        # 키워드별 추출된 데이터 리스트에 추가 (나중에 키워드별 통합 파일을 위해)
        keyword_specific_extracted_data[current_keyword].append(current_keyword_year_df)
        # 모든 키워드의 통합 저장을 위한 리스트에 추가
        all_extracted_sentences_for_combined_save.append(current_keyword_year_df)

        # 현재 키워드-연도 데이터를 별도 엑셀 파일로 저장
        output_keyword_year_path = os.path.join(base_path, f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx')
        try:
            if not current_keyword_year_df.empty:
                current_keyword_year_df[['언론사', '추출된_문장', '키워드', '연도', 'URL']].to_excel(output_keyword_year_path, index=False)
                print(f"'{current_keyword}' 키워드, {current_year}년 문장이 '{output_keyword_year_path}'에 성공적으로 저장되었습니다.")
            else:
                print(f"'{current_keyword}' 키워드, {current_year}년 추출된 문장이 없어 파일을 저장하지 않습니다.")
        except Exception as e:
            print(f"'{current_keyword}' 키워드, {current_year}년 엑셀 파일 저장 중 오류 발생: {e}")

### 4-3-7. '이슈' 키워드, 2024년 기사에서 문장 추출 및 파일 저장

In [ ]:
current_keyword = '이슈'
current_year = 2024

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(current_keyword, current_year, all_excel_files)

if df_keyword_year is None:
    print(f"경고: '{current_keyword}' 키워드에 대해 {current_year}년도 파일을 찾을 수 없습니다.")
else:
    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []
    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = df_keyword_year['URL'].dropna().unique().tolist()

    total_urls_in_excel = len(all_unique_urls_in_excel)

    if total_urls_in_excel == 0:
        print(f"경고: '{current_keyword}' 키워드, {current_year}년 파일에 'URL' 컬럼이 없거나 모든 URL이 비어있습니다. 건너뜨니다.")
    else:
        with tqdm(total=total_urls_in_excel, desc=f"'{current_keyword}' ({current_year}) 기사 처리 중") as pbar:
            for url in all_unique_urls_in_excel:
                import io
                import contextlib
                with contextlib.redirect_stdout(io.StringIO()):
                    article_text = fetch_article_content(url)

                if article_text:
                    sentences = find_sentences_with_keyword(article_text, current_keyword)
                    # '언론사'는 해당 URL이 원본 데이터에 여러 번 나타날 경우 불일치할 수 있으므로, 해당 URL과 일치하는 첫 번째 언론사를 사용합니다.
                    media = 'N/A'
                    matching_rows = df_keyword_year[df_keyword_year['URL'] == url]
                    if not matching_rows.empty and '언론사' in matching_rows.columns:
                        first_media = matching_rows['언론사'].dropna().iloc[0] if not matching_rows['언론사'].dropna().empty else 'N/A'
                        media = first_media

                    if sentences:
                        for sentence in sentences:
                            extracted_sentences_data_current_block.append({
                                '키워드': current_keyword,
                                '연도': current_year,
                                '언론사': media,
                                'URL': url,
                                '추출된_문장': sentence
                            })
                else:
                    failed_urls_current_block.append(url) # Store failed URL
                pbar.update(1)
                time.sleep(0.5)

        current_keyword_year_df = pd.DataFrame(extracted_sentences_data_current_block)

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 완료 ---")
        print(f"총 {len(current_keyword_year_df)}개의 '{current_keyword}' 키워드 문장이 {current_year}년 기사에서 추출되었습니다.")
        print("추출된 문장 DataFrame 미리보기:")
        display(current_keyword_year_df.head())

        # URL 처리 결과 요약
        successful_url_fetches = total_urls_in_excel - len(failed_urls_current_block)
        failed_url_fetches = len(failed_urls_current_block)
        failed_percentage = (failed_url_fetches / total_urls_in_excel * 100) if total_urls_in_excel > 0 else 0

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 URL 처리 결과 요약 ---")
        print(f"전체 엑셀 파일 내 고유 URL 개수: {total_urls_in_excel}개")
        print(f"기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): {successful_url_fetches}개")
        print(f"기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): {failed_url_fetches}개")
        print(f"실패율: {failed_percentage:.2f}%")


        # if failed_urls_current_block:
        #     print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL 목록 ({len(failed_urls_current_block)}개) ---")
        #     for failed_url in failed_urls_current_block:
        #         print(failed_url)
        # else:
        #     print(f"\n'{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL은 없습니다.")

        # 키워드별 추출된 데이터 리스트에 추가 (나중에 키워드별 통합 파일을 위해)
        keyword_specific_extracted_data[current_keyword].append(current_keyword_year_df)
        # 모든 키워드의 통합 저장을 위한 리스트에 추가
        all_extracted_sentences_for_combined_save.append(current_keyword_year_df)

        # 현재 키워드-연도 데이터를 별도 엑셀 파일로 저장
        output_keyword_year_path = os.path.join(base_path, f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx')
        try:
            if not current_keyword_year_df.empty:
                current_keyword_year_df[['언론사', '추출된_문장', '키워드', '연도', 'URL']].to_excel(output_keyword_year_path, index=False)
                print(f"'{current_keyword}' 키워드, {current_year}년 문장이 '{output_keyword_year_path}'에 성공적으로 저장되었습니다.")
            else:
                print(f"'{current_keyword}' 키워드, {current_year}년 추출된 문장이 없어 파일을 저장하지 않습니다.")
        except Exception as e:
            print(f"'{current_keyword}' 키워드, {current_year}년 엑셀 파일 저장 중 오류 발생: {e}")

### 4-3-8. '이슈' 키워드, 2025년 기사에서 문장 추출 및 파일 저장

In [ ]:
current_keyword = '이슈'
current_year = 2025

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(current_keyword, current_year, all_excel_files)

if df_keyword_year is None:
    print(f"경고: '{current_keyword}' 키워드에 대해 {current_year}년도 파일을 찾을 수 없습니다.")
else:
    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []
    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = df_keyword_year['URL'].dropna().unique().tolist()

    total_urls_in_excel = len(all_unique_urls_in_excel)

    if total_urls_in_excel == 0:
        print(f"경고: '{current_keyword}' 키워드, {current_year}년 파일에 'URL' 컬럼이 없거나 모든 URL이 비어있습니다. 건너뜨니다.")
    else:
        with tqdm(total=total_urls_in_excel, desc=f"'{current_keyword}' ({current_year}) 기사 처리 중") as pbar:
            for url in all_unique_urls_in_excel:
                import io
                import contextlib
                with contextlib.redirect_stdout(io.StringIO()):
                    article_text = fetch_article_content(url)

                if article_text:
                    sentences = find_sentences_with_keyword(article_text, current_keyword)
                    # '언론사'는 해당 URL이 원본 데이터에 여러 번 나타날 경우 불일치할 수 있으므로, 해당 URL과 일치하는 첫 번째 언론사를 사용합니다.
                    media = 'N/A'
                    matching_rows = df_keyword_year[df_keyword_year['URL'] == url]
                    if not matching_rows.empty and '언론사' in matching_rows.columns:
                        first_media = matching_rows['언론사'].dropna().iloc[0] if not matching_rows['언론사'].dropna().empty else 'N/A'
                        media = first_media

                    if sentences:
                        for sentence in sentences:
                            extracted_sentences_data_current_block.append({
                                '키워드': current_keyword,
                                '연도': current_year,
                                '언론사': media,
                                'URL': url,
                                '추출된_문장': sentence
                            })
                else:
                    failed_urls_current_block.append(url) # Store failed URL
                pbar.update(1)
                time.sleep(0.5)

        current_keyword_year_df = pd.DataFrame(extracted_sentences_data_current_block)

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 완료 ---")
        print(f"총 {len(current_keyword_year_df)}개의 '{current_keyword}' 키워드 문장이 {current_year}년 기사에서 추출되었습니다.")
        print("추출된 문장 DataFrame 미리보기:")
        display(current_keyword_year_df.head())

        # URL 처리 결과 요약
        successful_url_fetches = total_urls_in_excel - len(failed_urls_current_block)
        failed_url_fetches = len(failed_urls_current_block)
        failed_percentage = (failed_url_fetches / total_urls_in_excel * 100) if total_urls_in_excel > 0 else 0

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 URL 처리 결과 요약 ---")
        print(f"전체 엑셀 파일 내 고유 URL 개수: {total_urls_in_excel}개")
        print(f"기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): {successful_url_fetches}개")
        print(f"기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): {failed_url_fetches}개")
        print(f"실패율: {failed_percentage:.2f}%")


        # if failed_urls_current_block:
        #     print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL 목록 ({len(failed_urls_current_block)}개) ---")
        #     for failed_url in failed_urls_current_block:
        #         print(failed_url)
        # else:
        #     print(f"\n'{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL은 없습니다.")

        # 키워드별 추출된 데이터 리스트에 추가 (나중에 키워드별 통합 파일을 위해)
        keyword_specific_extracted_data[current_keyword].append(current_keyword_year_df)
        # 모든 키워드의 통합 저장을 위한 리스트에 추가
        all_extracted_sentences_for_combined_save.append(current_keyword_year_df)

        # 현재 키워드-연도 데이터를 별도 엑셀 파일로 저장
        output_keyword_year_path = os.path.join(base_path, f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx')
        try:
            if not current_keyword_year_df.empty:
                current_keyword_year_df[['언론사', '추출된_문장', '키워드', '연도', 'URL']].to_excel(output_keyword_year_path, index=False)
                print(f"'{current_keyword}' 키워드, {current_year}년 문장이 '{output_keyword_year_path}'에 성공적으로 저장되었습니다.")
            else:
                print(f"'{current_keyword}' 키워드, {current_year}년 추출된 문장이 없어 파일을 저장하지 않습니다.")
        except Exception as e:
            print(f"'{current_keyword}' 키워드, {current_year}년 엑셀 파일 저장 중 오류 발생: {e}")